# E29 · ScalarMix khusus tiap trait

Isi, Organisasi, Kebahasaan dan Mekanik memiliki empat bobot campuran layer
serta gamma masing-masing. Encoder RoBERTa hanya dijalankan sekali per batch.
Empat representasi hasil campuran masuk ke head trait terkait; attention
antartrait tetap memakai empat hidden head seperti E28.

Awalnya semua campuran 0,25 per layer dan gamma 1. E29 memiliki 20 parameter
campuran, tambahan 15 dibanding E28. Setelan training, split, seed dasar 44,
loss dan aturan checkpoint A/B mengikuti E28. Layer baru masuk optimizer LR 1e-4.

Pilih T4, hubungkan Drive ke workbook semi-final, lalu Run all. Lima fit
development langsung menampilkan metrik per fold. Fold selesai tersimpan.
ZIP 04–05 memuat empat metrik per trait, bobot layer/gamma tiap trait dan
perbandingan checkpoint B dengan E28 serta E25 dari arsip agregat run aktual.

Arsitektur ini adaptasi proyek dari campuran layer E43 dan head E25.
Manfaat tiap trait masih harus dibuktikan pada validasi.


## 1 · Setup pustaka pada kernel Colab

In [ ]:
# Jalankan pada runtime Colab baru, sebelum mengimpor pustaka numerik/model.
import sys
import subprocess
from importlib import metadata, util
from packaging.requirements import Requirement

# Pertahankan Torch/CUDA, NumPy, pandas, SciPy dan sklearn bawaan Colab.
# Mengganti pustaka numerik yang sudah dimuat kernel dapat menyebabkan konflik ABI.
CORE_REQUIREMENTS = {
    'torch': 'torch>=2.6', 'numpy': 'numpy>=1.26,<3',
    'pandas': 'pandas>=2,<3', 'scipy': 'scipy>=1.10,<2',
    'scikit-learn': 'scikit-learn>=1.2,<2',
}
for package, requirement in CORE_REQUIREMENTS.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError as error:
        raise RuntimeError('Gunakan runtime Google Colab baru; pustaka dasar belum ada: '+package) from error
    if not Requirement(requirement).specifier.contains(installed):
        raise RuntimeError(requirement+' diperlukan; runtime ini memiliki '+installed+'. Pilih runtime Colab baru.')

# Tokenizers 0.19 tidak menyediakan wheel Python 3.13; tetap satu versi per runtime.
TRANSFORMERS_REQUIREMENT = 'transformers==4.48.3' if sys.version_info >= (3, 13) else 'transformers==4.44.2'
EXTRA_REQUIREMENTS = [TRANSFORMERS_REQUIREMENT, 'openpyxl>=3.1,<4']
to_install = []
for requirement in EXTRA_REQUIREMENTS:
    req = Requirement(requirement)
    try:
        installed = metadata.version(req.name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or not req.specifier.contains(installed):
        if req.name in sys.modules:
            raise RuntimeError(req.name+' sudah diimpor dengan versi berbeda. Restart session, lalu Run all dari atas.')
        to_install.append(requirement)
if to_install:
    # Kendala ini mencegah pip mengganti pustaka numerik yang digunakan kernel.
    from pathlib import Path
    constraints = Path('/tmp/aes35_native_constraints.txt')
    constraints.write_text(''.join(p+'=='+metadata.version(p)+'\n' for p in CORE_REQUIREMENTS))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           '--constraint', str(constraints), *to_install])
print('Setup selesai. Semua cell berikutnya berjalan langsung di kernel notebook ini.')
print('Python:', sys.version.split()[0])


## 2 · Import dan ruang variabel notebook

In [ ]:
import os, sys, json, math, hashlib, random, warnings, re, itertools, unicodedata, zipfile, subprocess, gc, logging, html
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung PEFT.
    # Backend teks dikonfigurasi pada kernel notebook; tidak memuat operator vision.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from openpyxl import Workbook, load_workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
SCENARIOS = ('S3',)  # Hanya model fusion yang dijalankan.
NOTEBOOK_CODE_SHA256='06715f2272062d1164b8a06e0623a727bbad58dc885944f92eab175e4066f367'

from IPython.display import display, HTML


## 3 · Nama model pretrained

In [ ]:
# NAMA MODEL PRETRAINED — ubah di sini jika membuat eksperimen lain.
ROBERTA_CHECKPOINT = 'flax-community/indonesian-roberta-base'
BGE_CHECKPOINT = 'BAAI/bge-m3'

# Revision mengunci bobot dan tokenizer yang sama seperti E30.
ROBERTA_REVISION = 'cc5ecee7de8532880ad08d802292a6f42865bc83'
BGE_REVISION = '5617a9f61b028005a4858fdac845db406aefb181'

print('RoBERTa:', ROBERTA_CHECKPOINT, '| encoder ikut fine-tuning')
print('BGE-M3:', BGE_CHECKPOINT, '| encoder frozen; proyeksi/head tetap belajar')


## 4 · Pemetaan input dan empat target

In [ ]:
SCOPE = 'reset_e20_e19_checkpoint_ab'
SHEET_NAME = 'Sheet1'
QUESTION_COLUMN = 'Pertanyaan'
REFERENCE_COLUMN = 'Kunci_Jawaban'
ANSWER_COLUMN = 'Jawaban'
GROUP_COLUMN = 'Email Address'
RESPONSE_NUMBER_COLUMN = 'No_Soal'
INPUT_COLUMNS = (QUESTION_COLUMN, REFERENCE_COLUMN, ANSWER_COLUMN)

# Setel sesuai pedoman penilaian sebelum training. Rentang awal sesuai kategori di file contoh.
TRAITS = (
    ('isi', 'Skor_Isi', 1, 4),
    ('organisasi', 'Skor_Organisasi', 1, 2),
    ('kebahasaan', 'Skor_Tata_Bahasa', 1, 2),
    ('mekanik', 'Skor_Mekanik', 1, 2),
)
TARGET_COLUMNS = tuple(column for _, column, _, _ in TRAITS)
REQUIRED = (*INPUT_COLUMNS, GROUP_COLUMN, RESPONSE_NUMBER_COLUMN, *TARGET_COLUMNS)
RUBRIC = {'traits': [
    {'name': name, 'min': low, 'max': high, 'categories': list(range(low, high + 1))}
    for name, _, low, high in TRAITS
]}
SCORE_MIN = np.asarray([low for _, _, low, _ in TRAITS], dtype=np.float32)
SCORE_SPAN = np.asarray([high - low for _, _, low, high in TRAITS], dtype=np.float32)


## 5 · Hyperparameter E19

In [ ]:
# Setelan training E13. Satu perubahan ada pada bobot loss organisasi.
EPOCHS = 25
FOLDS = 5
EARLY_STOPPING_PATIENCE = 5
TRAIN_SEED = 44  # Satu konstanta pengacakan, tanpa loop multi-seed.
SPLIT_SEED = 2026
TEST_FRACTION = 0.20
MICRO_BATCH = 1
EFFECTIVE_BATCH = 8

ENCODER_LR = 5e-6             # Tetap E30; bukan LR E37 7e-6.
S3_MLP_LR = 1e-4              # MLP S3 tetap nilai asli E30.
S3_LAYERS_LR = 1e-4           # Kembali ke E30: BiGRU/co-attention/pooling/norm = 1e-4.
PROJECTION_LR = 1e-4          # Proyeksi BGE projection.* tidak ikut diturunkan.

DROPOUT = 0.1                   # Parameter lama untuk layer lain; tetap E30.
S3_MLP_DROPOUT = 0.2             # E3: dropout MLP E1 dinaikkan dari 0,1 ke 0,2.
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRADIENT_CLIP = 1.0

ROBERTA_MAX_TOKENS = 512
BGE_MAX_TOKENS = 1024
WINDOW_OVERLAP = 128
CONTEXT_MAX_TOKENS = 192
QUESTION_MAX_TOKENS = 64
DEVICE = 'cuda'



ORGANIZATION_LOSS_RATIO = 1.2
# Margin RMSE untuk kandidat checkpoint B, ditetapkan sebelum run.
CHECKPOINT_RMSE_MARGIN = 0.01


In [ ]:
print(f'Rasio bobot loss organisasi skor 1 terhadap skor 2: {ORGANIZATION_LOSS_RATIO:g}×')
print(f'Maksimum {EPOCHS} epoch per fold · patience {EARLY_STOPPING_PATIENCE} · '
      f'{FOLDS} fold · seed dasar {TRAIN_SEED}')
print('Ambang kategori organisasi tetap 1,50.')
print(f'Checkpoint B · margin RMSE {CHECKPOINT_RMSE_MARGIN:g} di atas A.')

### `config`

Periksa konfigurasi model, ukuran batch, dan anggaran token.

In [ ]:
# Nilai bawaan dan fungsi konfigurasi, tersedia langsung di notebook.

DEFAULTS = {
    'sheet': SHEET_NAME, 'expected_rows': None,
    'test_fraction': TEST_FRACTION, 'folds': FOLDS, 'seed': TRAIN_SEED, 'split_seed': SPLIT_SEED, 'split_candidates': 128,
    # Maksimal 25 epoch/fold; berhenti bila RMSE validasi tidak membaik selama patience.
    'epochs': EPOCHS, 'early_stopping_patience': EARLY_STOPPING_PATIENCE, 'effective_batch': EFFECTIVE_BATCH, 'micro_batch': MICRO_BATCH,
    'encoder_lr': ENCODER_LR, 's3_encoder_lr': ENCODER_LR, 'projection_lr': PROJECTION_LR, 'head_lr': 1e-4, 'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO, 'gradient_clip': GRADIENT_CLIP, 'dropout': DROPOUT, 's3_mlp_dropout': S3_MLP_DROPOUT,
    'roberta_max_length': ROBERTA_MAX_TOKENS, 'roberta_overlap': WINDOW_OVERLAP,
    'bge_max_length': BGE_MAX_TOKENS, 'bge_overlap': WINDOW_OVERLAP, 'bge_batch': 4, 'device': DEVICE,
    'roberta': {'checkpoint': ROBERTA_CHECKPOINT,
                'revision': ROBERTA_REVISION},
    'bge': {'checkpoint': BGE_CHECKPOINT,
            'revision': BGE_REVISION},
    # Batas konteks dihitung termasuk prefix; semua token jawaban tetap dicakup.
    'context_max_tokens': CONTEXT_MAX_TOKENS, 'question_max_tokens': QUESTION_MAX_TOKENS,
    'new_question': '', 'new_reference': '',
    'input_policy':'three_texts_all_scenarios_v1', 'input_format':'single_sequence',
    's3_mlp_lr': S3_MLP_LR, 's3_layers_lr': S3_LAYERS_LR,
    's3_fusion': 'concat', 
    'vector_source': 'pretrained', 'trace_position': 0,
}


# Gabungkan parameter pilihan dengan nilai bawaan, lalu periksa batas skor, batch, dan panjang input.
def config(values):
    result = {**DEFAULTS, **values}
    if not 0 < result['test_fraction'] < 1 or result['folds'] < 2 or result['split_candidates'] < 1:
        raise ValueError('Split/folds tidak sah')
    if result['epochs'] < 1 or not 1 <= result['micro_batch'] <= result['effective_batch']:
        raise ValueError('Epoch/batch tidak sah')
    if type(result['early_stopping_patience']) is not int or result['early_stopping_patience'] < 1:
        raise ValueError('early_stopping_patience harus bilangan bulat positif.')
    if result['vector_source'] not in ('pretrained', 'pretrained_pair', 'S3'):
        raise ValueError('vector_source harus pretrained, pretrained_pair, atau S3')
    if result['device'] not in ('auto', 'cpu', 'cuda'):
        raise ValueError('device harus auto, cpu, atau cuda')
    if not 0 <= result['warmup_ratio'] < 1:
        raise ValueError('warmup_ratio tidak sah')
    if any(not math.isfinite(result[k]) or result[k] <= 0
           for k in ('encoder_lr', 's3_encoder_lr', 'projection_lr', 'head_lr', 's3_mlp_lr', 's3_layers_lr', 'gradient_clip', 'bge_batch')):
        raise ValueError('Learning rate, gradient clip, dan batch BGE harus positif.')
    if not 0 <= result['s3_mlp_dropout'] < 1:
        raise ValueError('s3_mlp_dropout harus berada pada rentang [0,1).')
    if not 0 <= result['dropout'] < 1 or result['weight_decay'] < 0:
        raise ValueError('Dropout/weight decay tidak sah.')
    if not 2 < result['roberta_max_length'] <= 512 or not 0 <= result['roberta_overlap'] < result['roberta_max_length']-2:
        raise ValueError('Window RoBERTa melebihi batas atau overlap tidak sah.')
    if not 2 < result['bge_max_length'] <= 8192:
        raise ValueError('Batas BGE harus 3–8192 token termasuk special token.')
    if not (type(result['context_max_tokens']) is int and type(result['question_max_tokens']) is int
            and 1 <= result['question_max_tokens'] < result['context_max_tokens']
            and result['context_max_tokens']+4+result['roberta_overlap'] < result['roberta_max_length']):
        raise ValueError('Anggaran konteks harus menyisakan token jawaban lebih banyak daripada overlap.')
    if str(result.get('new_answer', '')).strip() and not all(str(result.get(k, '')).strip() for k in ('new_question', 'new_reference')):
        raise ValueError('Untuk NEW_ANSWER baru, isi juga NEW_QUESTION dan NEW_REFERENCE.')
    if not (type(result['bge_overlap']) is int and 0 <= result['bge_overlap']
            < result['bge_max_length']-result['context_max_tokens']-4):
        raise ValueError('Anggaran konteks BGE harus menyisakan kapasitas jawaban di atas overlap.')
    if result['input_policy']!='three_texts_all_scenarios_v1':
        raise ValueError('Notebook ini khusus tiga input pada semua skenario.')
    if result['input_format'] not in ('single_sequence','context_pair'):
        raise ValueError('Format input tidak sah.')
    if result['s3_fusion'] != 'concat':
        raise ValueError('Fusion/hidden gate tidak cocok dengan notebook ini.')
    return result


### `device_name`

Tentukan perangkat untuk encoder dan training.

In [ ]:
def device_name(cfg):
    requested = cfg['device']
    if requested == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA diminta tetapi tidak tersedia; pilih GPU di Colab atau device=cpu.')
    return ('cuda' if torch.cuda.is_available() else 'cpu') if requested == 'auto' else requested


### Periksa runtime

Tabel menunjukkan perangkat yang benar-benar tersedia di sesi ini.

In [ ]:
"""Native notebook runtime checks for data preparation and model execution."""


def runtime_information():
    available = torch.cuda.is_available()
    return {
        "torch": str(torch.__version__),
        "transformers": version("transformers"),
        "scikit-learn": version("scikit-learn"),
        "device": "cuda" if available else "cpu",
        "accelerator": torch.cuda.get_device_name(0) if available else None,
    }


def model_stage_ready(stage, required=()):
    """Report unavailable prerequisites before a model stage is executed."""
    missing_data = [
        name
        for name in ("cfg", "bundle", "manifest", "output")
        if name not in globals()
    ]
    if missing_data:
        print(
            f"{stage} belum dijalankan. Jalankan cell Siapkan dataset dan split "
            f'terlebih dahulu. Variabel yang diperlukan: {", ".join(missing_data)}.'
        )
        return False
    if cfg["device"] != "cpu" and not torch.cuda.is_available():
        print(
            f"{stage} belum dijalankan karena GPU CUDA belum tersedia. "
            "Dataset dan split sudah dapat diperiksa. Pilih Runtime → Change runtime type → T4 GPU, "
            "lalu Run all pada runtime GPU."
        )
        return False
    missing_stage = [name for name in required if name not in globals()]
    if missing_stage:
        print(
            f"{stage} belum dijalankan. Selesaikan tahap model sebelumnya. "
            f'Variabel yang diperlukan: {", ".join(missing_stage)}.'
        )
        return False
    return True


runtime_status = runtime_information()
display(pd.DataFrame([runtime_status]))
if not torch.cuda.is_available():
    print(
        "Runtime CPU terdeteksi. Pembacaan workbook dan pembagian data tetap berjalan. "
        'Tahap model menunggu GPU. Untuk uji CPU yang disengaja, set DEVICE = "cpu" '
        "di hyperparameter lalu jalankan ulang konfigurasi dari atas."
    )


## 6 · Dataset semi-final di Google Drive

### `read_dataset` dan `Bundle`

Workbook dibaca sebagai tiga kolom teks dan matriks empat skor. Email hanya dipakai untuk membentuk kelompok split.

In [ ]:
# Baca dan audit data tanpa mengubah file Excel asli.
# Rapikan Unicode dan karakter kontrol; ejaan, huruf besar, dan tanda baca jawaban tetap dipertahankan.
def clean(text, normalize_punctuation=False):
    text = unicodedata.normalize('NFC', text).replace('\r\n','\n').replace('\r','\n')
    text = ''.join(c for c in text if c in '\n\t' or unicodedata.category(c) != 'Cc')
    if normalize_punctuation:
        text = text.translate(str.maketrans({'“':'"','”':'"','‘':"'",'’':"'",'–':'-','—':'-'})).replace('…','...')
    return text

def digest(path):
    # Checkpoint besar diperiksa bertahap supaya tidak dimuat seluruhnya ke RAM.
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.pending')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+'\n', encoding='utf-8')
    temporary.replace(path)


@dataclass
class Bundle:
    rows: list
    y: np.ndarray
    groups: np.ndarray
    source_sha256: str
    summary: dict

    @property
    def texts(self):
        return [r['text'] for r in self.rows]


# Baca jawaban dan nilai guru, periksa kelengkapan, lalu kelompokkan email/duplikat untuk mencegah kebocoran data.
def read_dataset(cfg):
    path = Path(cfg['dataset'])
    source_hash = digest(path)
    book = load_workbook(path, read_only=True, data_only=True)
    try:
        if cfg['sheet'] not in book.sheetnames:
            raise ValueError(f"Tab {cfg['sheet']} tidak tersedia: {book.sheetnames}")
        data = iter(book[cfg['sheet']].iter_rows(values_only=True))
        header = [str(x).strip() if x is not None else None for x in next(data, ())]
        if any(header.count(x) != 1 for x in REQUIRED):
            raise ValueError('Header wajib hilang atau duplikat: '+', '.join(REQUIRED))
        rows = []
        for excel_row, values in enumerate(data, 2):
            if all(v is None for v in values):
                continue  # Hanya baris format kosong, bukan label/jawaban yang hilang.
            raw = {k: values[i] for i, k in enumerate(header) if k is not None}
            missing = [k for k in REQUIRED if raw[k] is None or not str(raw[k]).strip()]
            if missing:
                raise ValueError(f'Baris {excel_row}: kolom wajib kosong: '+', '.join(missing))
            scores = []
            for _, column, low, high in TRAITS:
                score = raw[column]
                if (isinstance(score, bool) or not isinstance(score, (int, float))
                    or not math.isfinite(score) or int(score) != score or not low <= score <= high):
                    raise ValueError(f'Baris {excel_row}: {column} harus bilangan bulat {low}–{high}.')
                scores.append(int(score))
            for key in (QUESTION_COLUMN,REFERENCE_COLUMN):
                if not isinstance(raw[key],str) or not raw[key].strip():
                    raise ValueError(f'Baris {excel_row}: {key} harus teks yang tidak kosong.')
            if not isinstance(raw[ANSWER_COLUMN], str) or not clean(raw[ANSWER_COLUMN]).strip():
                raise ValueError(f'Baris {excel_row}: jawaban bukan teks.')
            rows.append({'row_id': f'row_{excel_row-1:04d}', 'source_row': excel_row,
                         'raw_text': raw[ANSWER_COLUMN], 'text': clean(raw[ANSWER_COLUMN]),
                         'email_group': unicodedata.normalize('NFC', str(raw[GROUP_COLUMN])).strip().casefold(),
                         'question': clean(raw[QUESTION_COLUMN]), 'reference': clean(raw[REFERENCE_COLUMN]),
                         'response_number': str(raw[RESPONSE_NUMBER_COLUMN]), 'scores': scores})
    finally:
        book.close()
    if cfg.get('expected_rows') is not None and len(rows) != cfg['expected_rows']:
        raise ValueError(f"Diharapkan {cfg['expected_rows']} baris, ditemukan {len(rows)}; tidak dipotong otomatis.")
    if not rows:
        raise ValueError('Dataset kosong')
    parent = list(range(len(rows)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        parent[find(b)] = find(a)

    email_first, duplicate_sets = {}, {}
    for i, row in enumerate(rows):
        email = row['email_group']
        if email in email_first:
            union(email_first[email], i)
        email_first[email] = i
        # Normalisasi ini HANYA audit kesamaan, bukan teks masukan encoder.
        normalized = ' '.join(row['text'].casefold().split())
        key = hashlib.sha256(normalized.encode()).hexdigest()
        duplicate_sets.setdefault(key, []).append(i)
    for indices in duplicate_sets.values():
        for i in indices[1:]:
            union(indices[0], i)
    roots = {root: i for i, root in enumerate(sorted({find(i) for i in range(len(rows))}))}
    groups = np.asarray([roots[find(i)] for i in range(len(rows))], dtype=int)
    y = np.asarray([row.pop('scores') for row in rows], dtype=np.float32)
    summary = {
        'scope': SCOPE, 'rows': len(rows), 'target_columns': list(TARGET_COLUMNS),
        'model_outputs': len(TRAITS), 'categories': RUBRIC['traits'],
        'email_groups': len(email_first), 'connected_groups': len(roots),
        'questions': len({row['response_number'] for row in rows}),
        'duplicate_sets': sum(len(indices) > 1 for indices in duplicate_sets.values()),
        'source_sha256': source_hash, 'input_features': list(INPUT_COLUMNS),
    }
    for row in rows:
        del row['email_group']
    return Bundle(rows, y, groups, source_hash, summary)

# Identitas hasil mengikuti data, konfigurasi, dan kode notebook ini.
def run_folder(cfg, bundle):
    code_hash = NOTEBOOK_CODE_SHA256
    semantic = {k: v for k, v in cfg.items() if k not in ('dataset', 'output', 'trace_position', 'vector_source', 'new_answer', 'focus_word', 'new_question', 'new_reference')}
    key = hashlib.sha256(json.dumps({'source': bundle.source_sha256, 'config': semantic, 'code': code_hash}, sort_keys=True).encode()).hexdigest()[:16]
    folder = Path(cfg['output']) / key
    folder.mkdir(parents=True, exist_ok=True)
    dump(folder / 'effective_config.json', cfg)
    return folder


### storage

Cell ini menjalankan bagian storage.

In [ ]:
# Notebook eksperimen berikutnya bisa memakai path dan SHA-256 yang sama.
DATASET_VERSION = 'semi-final'
DATASET_NAME = 'data_cleaned_3sekolah.xlsx'
DATASET_SHA256 = 'dfbd97128b09a4eb488dd890ccc3efd9b9fe232e74fcc00ee5504a03acc58226'
DRIVE_MOUNT = Path('/content/drive')
DRIVE_DATASET = DRIVE_MOUNT / 'MyDrive' / 'AES' / 'datasets' / 'semi_final' / DATASET_NAME
LOCAL_DATASET = Path(DATASET_NAME)  # Untuk eksekusi di luar Colab.
OUTPUT_NAME = 'aes_reset_e29_scalar_mix_per_trait'
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
OUTPUT_ROOT = (DRIVE_MOUNT / 'MyDrive' / 'AES' / 'outputs' / OUTPUT_NAME
               if IN_COLAB else Path('outputs') / OUTPUT_NAME)


def resolve_colab_dataset(mount_drive, upload, drive_mount, drive_dataset):
    """Gunakan salinan privat Drive atau simpan workbook yang cocok pada upload pertama."""
    from tempfile import NamedTemporaryFile

    mount_drive(str(drive_mount))
    if not (drive_mount / 'MyDrive').is_dir():
        raise RuntimeError('MyDrive belum tersedia. Hubungkan Google Drive di Colab.')
    if not drive_dataset.exists():
        print(f'Unggah {DATASET_NAME} untuk penyimpanan pertama di Drive.')
        uploads = upload()
        if len(uploads) != 1:
            raise ValueError(f'Unggah satu workbook asli bernama {DATASET_NAME}.')
        workbook = next(iter(uploads.values()))
        if hashlib.sha256(workbook).hexdigest() != DATASET_SHA256:
            raise ValueError('File unggahan berbeda dari workbook semi-final yang dipakai eksperimen ini.')
        drive_dataset.parent.mkdir(parents=True, exist_ok=True)
        with NamedTemporaryFile(dir=drive_dataset.parent, suffix='.pending', delete=False) as stream:
            staging = Path(stream.name)
            try:
                stream.write(workbook)
                stream.close()
                if digest(staging) != DATASET_SHA256:
                    raise IOError('Salinan di Drive gagal diverifikasi.')
                if not drive_dataset.exists():
                    staging.replace(drive_dataset)
            finally:
                staging.unlink(missing_ok=True)
        print(f'Workbook semi-final tersedia di Drive: {drive_dataset}')
    return drive_dataset


if 'google.colab' in sys.modules or Path('/content').is_dir():
    from google.colab import drive, files
    DATASET_PATH = resolve_colab_dataset(drive.mount, files.upload, DRIVE_MOUNT, DRIVE_DATASET)
else:
    DATASET_PATH = LOCAL_DATASET

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Workbook semi-final belum ditemukan: {DATASET_PATH}')
if digest(DATASET_PATH) != DATASET_SHA256:
    raise ValueError(f'Workbook di {DATASET_PATH} berbeda dari versi semi-final terpilih. '
                     'Gunakan file asli atau tetapkan path dan SHA-256 versi baru secara eksplisit.')
print(f'Dataset {DATASET_VERSION} terverifikasi: {DATASET_PATH}')


## 7 · Baca workbook dan periksa X serta y

### Definisi split siswa

Fungsi ini tersedia sebelum dataset dan manifest disiapkan.

In [ ]:
def make_split(bundle, cfg):
    indices = np.arange(len(bundle.rows))
    splitter = GroupShuffleSplit(n_splits=cfg['split_candidates'], test_size=cfg['test_fraction'], random_state=cfg['split_seed'])
    # Pilih kedekatan JUMLAH BARIS dengan target 80:20, bukan skor/hasil model.
    # GroupShuffleSplit membagi kelompok; ukuran kelompok tidak selalu sama.
    candidates = list(splitter.split(indices, groups=bundle.groups))
    development, test = min(candidates, key=lambda pair: abs(len(pair[1])/len(indices)-cfg['test_fraction']))
    if len(set(bundle.groups[development])) < cfg['folds']:
        raise ValueError('Kelompok training tidak cukup untuk CV yang diminta.')
    folds = []
    for number, (a, b) in enumerate(GroupKFold(n_splits=cfg['folds']).split(development, groups=bundle.groups[development]), 1):
        training, validation = development[a], development[b]
        assert not set(bundle.groups[training]) & set(bundle.groups[validation])
        folds.append({'fold': number, 'train': training.tolist(), 'validation': validation.tolist()})
    assert not set(bundle.groups[development]) & set(bundle.groups[test])
    manifest = {'scope': SCOPE, 'source_sha256': bundle.source_sha256,
                'development': development.tolist(), 'test': test.tolist(), 'folds': folds,
                'development_rows': len(development), 'test_rows': len(test),
                'development_groups': len(set(bundle.groups[development])),
                'test_groups': len(set(bundle.groups[test])),
                'target_test_row_fraction': cfg['test_fraction'],
                'candidate_group_fraction': cfg['test_fraction'],
                'split_candidates': cfg['split_candidates'],
                'selection_uses_labels': False,
                'actual_test_row_fraction': len(test)/len(indices),
                'policy': 'Holdout kelompok: pilih kandidat deterministik terdekat target proporsi baris berdasarkan ukuran kelompok saja, tanpa label/hasil model. CV hanya di development. Email/duplikat tidak menyeberang. Mengukur kelompok baru pada pertanyaan ini, bukan unseen-prompt.',
                'test_used_for_selection': False}
    manifest['split_sha256'] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    return manifest


In [ ]:
cfg = config({
    'dataset': str(DATASET_PATH), 'output': str(OUTPUT_ROOT),
    'experiment': 'Reset_E29_S3_ScalarMix_PerTrait', 'head_type': 'independent_trait_heads', 'trait_hidden': 64, 'series': 'multitrait_reset', 'dataset_version': DATASET_VERSION, 'traits': TRAITS,
    'runtime': runtime_information(), 'isi_loss_ratio': 1.0, 'organization_loss_ratio': ORGANIZATION_LOSS_RATIO,
    'checkpoint_rmse_margin': CHECKPOINT_RMSE_MARGIN,
    'fusion_variant': 'e25_scalar_mix_per_trait',
    'scalar_mix_layers': 4, 'scalar_mix_per_trait': True, 'scalar_mix_gamma_initial': 1.0, 'trait_corr_threshold': 0.7, 'trait_relation_weight': 0.0,
})
bundle = read_dataset(cfg)
print(f"{len(bundle.rows)} jawaban, {bundle.summary['email_groups']} akun, "
      f"{bundle.summary['connected_groups']} kelompok setelah penggabungan duplikat.")
print(f"X berisi {len(INPUT_COLUMNS)} kolom teks: {', '.join(INPUT_COLUMNS)}")
print(f"y berukuran {bundle.y.shape}")
display(pd.DataFrame([{
    'trait': name, 'kolom_target': column, 'minimum': low, 'maksimum': high,
} for name, column, low, high in TRAITS]))

# Selesaikan dataset dan split pada cell yang sama.
manifest = make_split(bundle, cfg)
E25_SEED44_REFERENCE = {'split_sha256': 'f7fba02006ae61f96fa586fb6b87de678038b38679ac0629af101cb66e69c16c', 'seed': 44, 'zip_sha256': '2793ccfa066c744acb0d2d010f08a20e81bf3bb9d641849de7b13481a29815ad', 'metrics': [{'trait': 'isi', 'RMSE': 0.6122827180744828, 'Pearson': 0.6319319338036563, 'QWK': 0.5900245463204943, 'Gwet AC2': 0.8741512499826231}, {'trait': 'organisasi', 'RMSE': 0.329305108581519, 'Pearson': 0.4652458661776662, 'QWK': 0.4364718607067872, 'Gwet AC2': 0.8150486565998906}, {'trait': 'kebahasaan', 'RMSE': 0.3642530181411199, 'Pearson': 0.48967923106631, 'QWK': 0.472667535263128, 'Gwet AC2': 0.7651904784726409}, {'trait': 'mekanik', 'RMSE': 0.3463042238132857, 'Pearson': 0.5891276612171741, 'QWK': 0.5703713423435245, 'Gwet AC2': 0.7857284493223764}], 'overview': {'konfigurasi': 'E25 kontrol 1x', 'RMSE rata-rata fold': 0.428445306213117, 'macro QWK rata-rata fold': 0.5117242469275849, 'label isi 1': 17, 'isi 1 terdeteksi': 1, 'label 2–4 ke 1': 1}}
if manifest['split_sha256'] != E25_SEED44_REFERENCE['split_sha256']:
    raise ValueError('Split E28 berbeda dari E25 seed 44. Periksa dataset dan versi scikit-learn.')
if cfg['seed'] != E25_SEED44_REFERENCE['seed']:
    raise ValueError('Seed E28 harus sama dengan E25 seed 44.')
E28_REFERENCE = {'split_sha256': 'f7fba02006ae61f96fa586fb6b87de678038b38679ac0629af101cb66e69c16c', 'seed': 44, 'source_sha256': 'fa561f8f758f4899d2f339d7bcfbd0b56f1dde7aa33d56808595b01c41bf483a', 'metrics': [{'trait': 'isi', 'RMSE': 0.6001123162942513, 'Pearson': 0.6469596269291701, 'QWK': 0.6131562114708964, 'Gwet AC2': 0.8859096244559314}, {'trait': 'organisasi', 'RMSE': 0.3185225873957397, 'Pearson': 0.4884452602444983, 'QWK': 0.4570255695588947, 'Gwet AC2': 0.8373173820102843}, {'trait': 'kebahasaan', 'RMSE': 0.3701622044227915, 'Pearson': 0.4777849003489264, 'QWK': 0.470314908362679, 'Gwet AC2': 0.7616956659285037}, {'trait': 'mekanik', 'RMSE': 0.3480008321615385, 'Pearson': 0.5842865210420713, 'QWK': 0.5743582851767235, 'Gwet AC2': 0.7952492022528278}]}
E25_REFERENCE = {'split_sha256': 'f7fba02006ae61f96fa586fb6b87de678038b38679ac0629af101cb66e69c16c', 'seed': 44, 'source_sha256': '2793ccfa066c744acb0d2d010f08a20e81bf3bb9d641849de7b13481a29815ad', 'metrics': [{'trait': 'isi', 'RMSE': 0.6122827180744828, 'Pearson': 0.6319319338036563, 'QWK': 0.5900245463204943, 'Gwet AC2': 0.8741512499826231}, {'trait': 'organisasi', 'RMSE': 0.329305108581519, 'Pearson': 0.4652458661776662, 'QWK': 0.4364718607067872, 'Gwet AC2': 0.8150486565998906}, {'trait': 'kebahasaan', 'RMSE': 0.3642530181411199, 'Pearson': 0.48967923106631, 'QWK': 0.472667535263128, 'Gwet AC2': 0.7651904784726409}, {'trait': 'mekanik', 'RMSE': 0.3463042238132857, 'Pearson': 0.5891276612171741, 'QWK': 0.5703713423435245, 'Gwet AC2': 0.7857284493223764}]}
if any(ref['split_sha256'] != manifest['split_sha256'] or ref['seed'] != cfg['seed']
       for ref in (E28_REFERENCE, E25_REFERENCE)):
    raise ValueError('Split dan seed harus cocok dengan E28 dan E25.')
output = run_folder(cfg, bundle)
dump(output / 'split.json', manifest)
print(f"Split siap: {len(manifest['development'])} development, {len(manifest['test'])} test.")

print(f'Folder penyimpanan run: {output}')


## 8 · Pisahkan development, test, dan lima fold

### `contexts_from_bundle`

Ambil pasangan pertanyaan dan referensi yang mengiringi setiap jawaban.

In [ ]:
def contexts_from_bundle(bundle):
    """Ambil soal/referensi tanpa membaca target skor guru."""
    contexts = []
    for row in bundle.rows:
        question, reference = row.get('question'), row.get('reference')
        if not isinstance(question, str) or not question.strip():
            raise ValueError('Context pair memerlukan soal yang tidak kosong.')
        if not isinstance(reference, str) or not reference.strip():
            raise ValueError('Context pair memerlukan jawaban referensi yang tidak kosong.')
        contexts.append((clean(question), clean(reference)))
    return contexts


### split execute

Cell ini menjalankan bagian split execute.

In [ ]:
if all(name in globals() for name in ('cfg', 'bundle', 'manifest', 'output')):
    print(f"Development: {len(manifest['development'])} | Test: {len(manifest['test'])}")
    display(pd.DataFrame([{
        'fold': f['fold'], 'training': len(f['train']), 'validation': len(f['validation']),
    } for f in manifest['folds']]))
else:
    print('Jalankan cell Siapkan dataset dan split terlebih dahulu.')


## 9 · Tokenisasi, window, dan posisi input

### `mean_by_essay`

Rata-ratakan vektor window yang berasal dari jawaban yang sama.

In [ ]:
def mean_by_essay(cls, mapping, n):
    sums=cls.new_zeros((n,cls.shape[-1])).index_add(0,mapping,cls)
    counts=cls.new_zeros(n).index_add(0,mapping,cls.new_ones(len(mapping)))
    if (counts==0).any(): raise ValueError('Esai tanpa window')
    return sums / counts[:,None]


### `context_token_ids`

Ubah soal dan referensi jadi ID token sesuai batas konteks.

In [ ]:
def context_token_ids(context, tokenizer, cfg):
    """Sisakan anggaran untuk soal dan referensi sekaligus."""
    if not isinstance(context, (tuple, list)) or len(context) != 2:
        raise ValueError('Konteks harus pasangan (soal, jawaban referensi).')
    question, reference = context
    if not all(isinstance(t, str) and t.strip() for t in context):
        raise ValueError('Isi soal dan jawaban referensi sebelum memprediksi.')
    limit, qlimit = cfg['context_max_tokens'], cfg['question_max_tokens']
    if not 1 <= qlimit < limit:
        raise ValueError('Anggaran token soal harus lebih kecil daripada konteks.')
    # Prefix menjadi bagian input; nilai guru tidak pernah dimasukkan.
    qids = tokenizer.encode('Soal: '+question, add_special_tokens=False)
    rids = tokenizer.encode('\nJawaban referensi: '+reference, add_special_tokens=False)
    question_kept = qids[:qlimit]
    reference_kept = rids[:limit-len(question_kept)]
    return question_kept+reference_kept, {
        'question_tokens': len(qids), 'question_kept': len(question_kept),
        'reference_tokens': len(rids), 'reference_kept': len(reference_kept),
        'question_truncated': len(question_kept) < len(qids),
        'reference_truncated': len(reference_kept) < len(rids),
        'context_tokens_kept': len(question_kept)+len(reference_kept),
    }


### `reference_context_positions`

Tandai posisi token referensi untuk co-attention.

In [ ]:
def reference_context_positions(context, tok, cfg, info):
    """Posisi isi referensi dari offset karakter; ID token berulang tidak ditebak."""
    prefix = '\nJawaban referensi: '
    text = prefix + context[1]
    try:
        aligned = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    except NotImplementedError:
        # RoBERTa training tetap slow seperti E26; fast hanya membaca offset.
        if not hasattr(tok, '_aes_reference_fast'):
            spec = cfg['roberta']
            tok._aes_reference_fast = AutoTokenizer.from_pretrained(
                spec['checkpoint'], revision=spec['revision'], use_fast=True)
        aligned = tok._aes_reference_fast(text, add_special_tokens=False, return_offsets_mapping=True)
    original = tok.encode(text, add_special_tokens=False)
    if list(aligned['input_ids']) != original or len(aligned['offset_mapping']) != len(original):
        raise ValueError('ID/offset tokenizer referensi tidak identik; posisi tidak ditebak.')
    positions = []
    for index, (start, end) in enumerate(aligned['offset_mapping'][:info['reference_kept']]):
        if not 0 <= start <= end <= len(text):
            raise ValueError('Offset tokenizer referensi tidak sah.')
        # Token yang mencakup teks isi diperbolehkan; token prefix/spasi saja dikecualikan.
        if end > len(prefix) and text[max(start, len(prefix)):end].strip():
            positions.append(info['question_kept'] + index)
    if not positions:
        raise ValueError('Isi referensi habis oleh truncation/prefix. Tambah anggaran konteks sebelum training.')
    return positions


### `pair_content_windows`

Rangkai tiga teks dan bagi jawaban panjang menjadi window yang bertumpang tindih.

In [ ]:
def pair_content_windows(text, context, tokenizer, maximum=512, overlap=128, cfg=None):
    """Informasi dan window sama; E22 menambah separator resmi pasangan."""
    if cfg is None or cfg.get('input_format') not in ('single_sequence', 'context_pair'):
        raise ValueError('Pilih format input E21 atau E22 sebelum tokenisasi.')
    context_ids, context_info = context_token_ids(context, tokenizer, cfg)
    reference_positions = reference_context_positions(context, tokenizer, cfg, context_info)
    context_layout = [-1]*len(context_ids)
    for position in reference_positions:context_layout[position] = -4
    ids = tokenizer.encode(text, add_special_tokens=False)
    answer_prefix = tokenizer.encode('\nJawaban siswa:\n', add_special_tokens=False)
    if not ids:
        raise ValueError('Jawaban tidak menghasilkan token.')
    paired = cfg['input_format'] == 'context_pair'
    specials = tokenizer.num_special_tokens_to_add(pair=paired)
    # Cadangkan overhead terbesar untuk KEDUA eksperimen. Starts/cakupan sama.
    reserved = max(tokenizer.num_special_tokens_to_add(pair=False),
                   tokenizer.num_special_tokens_to_add(pair=True))
    capacity = maximum-len(context_ids)-len(answer_prefix)-reserved
    if not 0 <= overlap < capacity:
        raise ValueError('Konteks dan penanda jawaban menyisakan kapasitas di bawah overlap.')
    starts = [0]
    while starts[-1]+capacity < len(ids):
        starts.append(starts[-1]+capacity-overlap)
    chunks, answer_positions, reference_window_positions, segments = [], [], [], []
    for start in starts:
        answer = ids[start:start+capacity]
        second = answer_prefix+answer
        if paired:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids, second)
            layout = tokenizer.build_inputs_with_special_tokens(context_layout,
                     [-3]*len(answer_prefix)+[-2]*len(answer))
        else:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids+second)
            layout = tokenizer.build_inputs_with_special_tokens(
                     context_layout+[-3]*len(answer_prefix)+[-2]*len(answer))
        positions = [i for i, value in enumerate(layout) if value == -2]
        if (len(prepared) != len(layout) or len(prepared) > maximum
                or len(prepared)-len(context_ids)-len(second) != specials
                or [prepared[i] for i in positions] != answer):
            raise ValueError('Template tokenizer tidak konsisten dengan format input.')
        chunks.append({'input_ids':prepared, 'attention_mask':[1]*len(prepared)})
        answer_positions.append(positions)
        reference_window_positions.append([i for i, value in enumerate(layout) if value == -4])
        segments.append(['Soal/referensi' if v == -1 else 'Jawaban siswa' if v == -2
                         else 'Penanda jawaban' if v == -3 else 'Isi referensi' if v == -4 else 'Token khusus' for v in layout])
    stats = {**context_info, 'input_format':cfg['input_format'], 'content_tokens':len(ids),
             'capacity':capacity, 'stride':capacity-overlap, 'starts':starts,
             'windows':len(chunks), 'pair_special_tokens':specials,
             'reserved_special_tokens':reserved, 'answer_prefix_tokens':len(answer_prefix),
             'answer_positions':answer_positions, 'reference_positions':reference_window_positions,
             'reference_content_kept':len(reference_positions), 'segments':segments, 'answer_tokens_uncovered':0}
    return chunks, stats


### `pair_batch_windows`

Tambahkan padding dan simpan pemetaan window ke jawaban.

In [ ]:
def pair_batch_windows(texts, contexts, tokenizer, device='cpu', maximum=512, overlap=128, cfg=None):
    """Susun batch pasangan tanpa menukar konteks antarjawaban."""
    if len(texts) != len(contexts) or not texts:
        raise ValueError('Jumlah konteks harus sama dengan jumlah jawaban.')
    chunks, mapping, stats = [], [], []
    for index, (text, context) in enumerate(zip(texts, contexts)):
        values, detail = pair_content_windows(text, context, tokenizer, maximum, overlap, cfg)
        chunks.extend(values)
        mapping.extend([index]*len(values))
        stats.append(detail)
    batch = tokenizer.pad(chunks, padding=True, return_tensors='pt')
    return {key: value.to(device) for key, value in batch.items()}, torch.tensor(mapping, device=device), stats


## 10 · Memuat tokenizer dan encoder pretrained

### `release`

Bebaskan alokasi yang selesai dipakai sebelum berpindah tahap.

In [ ]:
def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


### ExpectedPoolerNotice

Cell ini menjalankan bagian ExpectedPoolerNotice.

In [ ]:
class ExpectedPoolerNotice(logging.Filter):
    """Hanya saring pemberitahuan dua bobot pooler yang memang tidak dipakai."""
    def filter(self, record):
        import ast
        import re
        message = record.getMessage()
        if not message.startswith('Some weights of the model checkpoint at ') or 'were not used when initializing' not in message:
            return True
        match = re.search(r"\['[^\n]+?\]", message)
        if match is None:
            return True
        try:
            keys = ast.literal_eval(match.group())
        except (ValueError, SyntaxError):
            return True
        return set(keys) != {'pooler.dense.bias', 'pooler.dense.weight'}


### `encoder`

Muat bobot pretrained. Jalur BGE dibekukan dan RoBERTa disiapkan untuk fine-tuning.

In [ ]:
def encoder(cfg, role):
    # Ambil nama checkpoint dan revision model yang dipilih: roberta atau bge.
    spec = cfg[role]
    logger = logging.getLogger('transformers.modeling_utils')
    notice = ExpectedPoolerNotice()
    logger.addFilter(notice)
    try:
        # Muat arsitektur dan bobot pretrained. Ini baru inisialisasi encoder, belum latihan memakai nilai guru.
        result = AutoModel.from_pretrained(spec['checkpoint'], revision=spec['revision'], add_pooling_layer=False)
    finally:
        logger.removeFilter(notice)
    # Periksa panjang vektor: RoBERTa 768 angka, BGE-M3 1024 angka.
    expected = 768 if role == 'roberta' else 1024
    if result.config.hidden_size != expected:
        raise ValueError(f'{role}: dimensi checkpoint tidak cocok dengan {expected}.')
    # Parameter RoBERTa ikut diperbarui; parameter BGE tidak ikut diperbarui.
    # RoBERTa boleh belajar; bobot encoder BGE tidak diperbarui.
    result.requires_grad_(role == 'roberta')
    return result


### `tokenizer`

Muat tokenizer dari repositori dan revisi yang cocok dengan encoder.

In [ ]:
def tokenizer(cfg, role, fast=False):
    # Muat tokenizer dari checkpoint yang sama agar ID token cocok dengan kosakata model.
    result = AutoTokenizer.from_pretrained(cfg[role]['checkpoint'], revision=cfg[role]['revision'],
                                           use_fast=fast if role == 'roberta' else True)
    if role == 'roberta' and not fast and result.num_special_tokens_to_add(False) != 2:
        raise ValueError('RoBERTa slow tokenizer harus menambahkan dua special token.')
    return result


### Cek window pada satu jawaban development

Tabel menampilkan panjang input dan jumlah token yang ditandai untuk referensi serta jawaban.

In [ ]:
if model_stage_ready('Cek tokenisasi', ()):
    # Ambil satu jawaban development dan perlihatkan ukuran window serta posisi yang dipakai E43.
    example_index = manifest['development'][0]
    example_row = bundle.rows[example_index]
    example_context = (example_row['question'], example_row['reference'])
    example_tokenizer = tokenizer(cfg, 'roberta')
    example_windows, example_detail = pair_content_windows(
        example_row['text'], example_context, example_tokenizer,
        cfg['roberta_max_length'], cfg['roberta_overlap'], cfg,
    )
    display(pd.DataFrame([{
        'id_jawaban': example_row['row_id'],
        'window': n,
        'panjang_input': len(window['input_ids']),
        'token_referensi': len(reference),
        'token_jawaban': len(answer),
    } for n, (window, reference, answer) in enumerate(zip(
        example_windows, example_detail['reference_positions'], example_detail['answer_positions'],
    ), 1)]))


## 11 · Fitur BGE-M3 frozen

### `paired_pretrained_values`

Gabungkan vektor dari semua window menjadi satu vektor per jawaban.

In [ ]:
def paired_pretrained_values(bundle, cfg, role):
    """Mean CLS antar-window; BGE dinormalisasi L2 setelah agregasi."""
    device = device_name(cfg)
    model, tok = encoder(cfg,role).to(device), tokenizer(cfg,role)
    model.eval()
    vectors, window_vectors, ids, coverage = [], [], [], []
    with torch.no_grad():
        for index, (text,context) in enumerate(zip(bundle.texts,contexts_from_bundle(bundle))):
            windows,mapping,details = pair_batch_windows([text],[context],tok,device,
                cfg[role+'_max_length'],cfg[role+'_overlap'],cfg)
            # Satu jawaban diproses bertahap agar jumlah window tidak meledakkan memori.
            cls_parts=[]
            for start in range(0,len(mapping),cfg['bge_batch']):
                part={k:v[start:start+cfg['bge_batch']] for k,v in windows.items()}
                cls_parts.append(model(**part).last_hidden_state[:,0,:].float())
            cls=torch.cat(cls_parts,dim=0)
            value=mean_by_essay(cls,mapping,1)
            if role=='bge':value=torch.nn.functional.normalize(value,dim=1)
            vectors.append(value.cpu().numpy())
            window_vectors.append(cls.cpu().numpy())
            row_id=bundle.rows[index].get('row_id',f'inference_{index}')
            ids.extend(f'{row_id}_window_{j}' for j in range(len(cls)))
            coverage.append({k:v for k,v in details[0].items() if k not in ('answer_positions','reference_positions','segments')})
            if (index+1)%25==0 or index+1==len(bundle.rows):
                print(f'{role}: tiga input {index+1}/{len(bundle.rows)} jawaban selesai.',flush=True)
    values=np.concatenate(vectors).astype(np.float32)
    if role=='bge' and any(p.requires_grad for p in model.parameters()):
        raise ValueError('Encoder BGE harus frozen.')
    del model,tok
    release()
    return values,np.concatenate(window_vectors),ids,coverage


### `bge_features`

Pakai cache bila teks dan konfigurasi tokenizer sama; vektor berdimensi 1024.

In [ ]:
def bge_features(bundle, cfg, output):
    """Cache berubah jika soal, referensi, jawaban, atau resep token berubah."""
    contexts=contexts_from_bundle(bundle)
    contract={'model':cfg['bge'], 'texts':bundle.texts, 'contexts':contexts,
              'maximum':cfg['bge_max_length'],'overlap':cfg['bge_overlap'],
              'context_max_tokens':cfg['context_max_tokens'],'question_max_tokens':cfg['question_max_tokens'],
              'policy':'three_inputs_mean_CLS_then_L2_v2', 'input_format':cfg['input_format']}
    key=hashlib.sha256(json.dumps(contract,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:20]
    cache=Path(cfg['output'])/'cache'/f'bge_three_inputs_{key}.npz'
    metadata=cache.with_suffix('.json')
    if cache.exists() and metadata.exists():
        proof=json.loads(metadata.read_text())
        if proof['file_sha256']!=hashlib.sha256(cache.read_bytes()).hexdigest():
            raise ValueError('Cache BGE berubah; tidak dipakai.')
        with np.load(cache,allow_pickle=False) as data:values=data['features']
        reused=True
    else:
        values,_,_,coverage=paired_pretrained_values(bundle,cfg,'bge')
        cache.parent.mkdir(parents=True,exist_ok=True)
        np.savez_compressed(cache,features=values)
        proof={'model':cfg['bge'],'shape':list(values.shape),'frozen':True,
               'pooling':'mean paired-window CLS then L2','max_length':cfg['bge_max_length'],
               'feature_source':'Pertanyaan + Kunci_Jawaban + Jawaban', 'input_format':cfg['input_format'],
               'answer_tokens_uncovered':sum(s['answer_tokens_uncovered'] for s in coverage),
               'context_truncated_rows':sum(s['question_truncated'] or s['reference_truncated'] for s in coverage),
               'file_sha256':hashlib.sha256(cache.read_bytes()).hexdigest()}
        dump(metadata,proof)
        reused=False
    if values.shape!=(len(bundle.rows),1024) or not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values,axis=1),1,atol=1e-4):
        raise ValueError('Fitur BGE shape/normalisasi tidak sah.')
    dump(Path(output)/'bge_encoding.json',{**proof,'reused':reused})
    return torch.from_numpy(values)


### Hitung fitur BGE untuk seluruh jawaban

Cell ini menampilkan progres encoding dan ukuran matriks vektor yang akan dipakai pada CV dan refit.

In [ ]:
if model_stage_ready('Encoding BGE', ()):
    development = manifest['development']
    dev_bundle = Bundle([bundle.rows[i] for i in development],
                        bundle.y[development], bundle.groups[development],
                        bundle.source_sha256, bundle.summary)
    dev_features = bge_features(dev_bundle, cfg, output)
    # Pertahankan indeks workbook asli yang dipakai fold.
    bge = dev_features.new_full((len(bundle.rows), dev_features.shape[1]), float('nan'))
    bge[development] = dev_features
    coverage = json.loads((output / 'bge_encoding.json').read_text())
    display(pd.DataFrame([{
        'encoder_frozen': cfg['bge']['checkpoint'],
        'vektor_development': len(dev_features), 'dimensi': bge.shape[1],
        'cache_dipakai_ulang': coverage['reused'],
        'token_jawaban_tidak_tercakup': coverage['answer_tokens_uncovered'],
    }]))


## 12 · Arsitektur E25 dengan ScalarMix-4

In [ ]:
class ScalarMix(nn.Module):
    """Campurkan k layer Transformer terakhir dengan bobot global yang dipelajari."""
    def __init__(self, number_of_layers=4, gamma_initial=1.0):
        super().__init__()
        if type(number_of_layers) is not int or number_of_layers < 1:
            raise ValueError('Jumlah layer Scalar Mix harus bilangan bulat positif.')
        self.number_of_layers = number_of_layers
        # Empat logit nol memberi bobot0.25; gamma awal1 mempertahankan keluaran awal E41.
        self.logits = nn.Parameter(torch.zeros(number_of_layers))
        self.gamma = nn.Parameter(torch.full((1,), float(gamma_initial)))
        self.capture_stats = False

    def weights(self):
        return torch.softmax(self.logits, dim=0)

    def forward(self, hidden_states, attention_mask=None):
        # hidden_states[0] adalah embeddings; hanya keluaran layer Transformer dipilih.
        if hidden_states is None or len(hidden_states)-1 < self.number_of_layers:
            raise ValueError('Encoder tidak menyediakan cukup layer Transformer untuk Scalar Mix.')
        selected = hidden_states[-self.number_of_layers:]
        if any(value.shape != selected[0].shape for value in selected):
            raise ValueError('Semua tensor layer Scalar Mix harus mempunyai shape yang sama.')
        weights = self.weights().to(dtype=selected[0].dtype)
        # Penjumlahan bertahap menghindari stack tambahan [4,batch,token,768].
        mixed = torch.zeros_like(selected[0])
        for weight, value in zip(weights, selected):
            mixed = mixed + weight * value
        scaled = self.gamma.to(dtype=mixed.dtype) * mixed
        # Statistik hanya saat inspeksi fitur, tanpa graph gradien atau sinkronisasi CPU tiap train step.
        if self.capture_stats:
            with torch.no_grad():
                valid = attention_mask.bool() if attention_mask is not None else torch.ones_like(mixed[..., 0], dtype=torch.bool)
                counts = valid.sum(1).clamp_min(1)
                def average_norm(value):
                    return ((value.float().norm(dim=-1)*valid).sum(1)/counts).unsqueeze(1)
                self.norm_before = average_norm(mixed)
                self.norm_after = average_norm(scaled)
                self.norm_last_layer = average_norm(hidden_states[-1])
        return scaled

    def report(self, encoder_layers):
        weights = self.weights().detach().float().cpu().tolist()
        logits = self.logits.detach().float().cpu().tolist()
        start = encoder_layers-self.number_of_layers
        return [{'Indeks Layer RoBERTa': start+i, 'Layer ke-': start+i+1,
                 'Logit': logits[i], 'Bobot Softmax': weights[i],
                 'Bobot Awal': 1/self.number_of_layers, 'Gamma': float(self.gamma.detach().cpu().item())}
                for i in range(self.number_of_layers)]




class IndependentTraitHeads(nn.Module):
    """Four independent 1024→64→1 scorers, in the declared trait order."""

    def __init__(self, dropout):
        super().__init__()
        self.heads = nn.ModuleDict({
            name: nn.Sequential(
                nn.Linear(1024, 64),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(64, 1),
            )
            for name, *_ in TRAITS
        })

    def forward(self, fusion):
        # Input features come from text. The four teacher scores enter the loss.
        return torch.cat(
            [self.heads[name](fusion) for name, *_ in TRAITS], dim=1
        )

class PromptAwareHeads(IndependentTraitHeads):
    def __init__(self, dropout):
        super().__init__(dropout)
        self.question_query = nn.Linear(768, 4 * 64, bias=False)
        self.answer_key = nn.Linear(768, 64, bias=False)
        self.prompt_residual = nn.Linear(768, 64, bias=False)
        self.trait_residual = nn.Linear(64, 64, bias=False)
        nn.init.zeros_(self.prompt_residual.weight)
        nn.init.zeros_(self.trait_residual.weight)


    def trait_state(self, fusion, tokens, question_mask, answer_mask, mapping, index):
        name = TRAITS[index][0]
        linear, activation, dropout, _ = self.heads[name]
        state = linear(fusion)
        if name != 'organisasi':
            prompt = (tokens * question_mask.unsqueeze(-1)).sum(1) / question_mask.sum(1, keepdim=True)
            query = self.question_query(prompt).reshape(len(tokens), 4, 64)[:, index:index + 1]
            keys = self.answer_key(tokens)
            logits = query.float() @ keys.float().transpose(1, 2) / 8
            weights = logits.masked_fill(~answer_mask[:, None, :], float('-inf')).softmax(-1)
            pooled = (weights.to(tokens.dtype) @ tokens).squeeze(1)
            context = mean_by_essay(pooled, mapping, len(fusion))
            state = state + self.prompt_residual(context)
        return dropout(activation(state))

    def finish(self, states):
        hidden = torch.stack(states, dim=1)
        logits = hidden.float() @ hidden.float().transpose(1, 2) / 8
        diagonal = torch.eye(4, dtype=torch.bool, device=hidden.device)
        weights = logits.masked_fill(diagonal, float('-inf')).softmax(-1)
        updated = hidden + self.trait_residual(weights.to(hidden.dtype) @ hidden)
        return torch.cat([self.heads[name][3](updated[:, i])
                          for i, (name, *_) in enumerate(TRAITS)], dim=1)


def prompt_answer_masks(details, width, padding_side, device):
    question, answer = [], []
    for info in details:
        for segments, positions in zip(info['segments'], info['answer_positions']):
            prompt_positions = [i for i, label in enumerate(segments)
                                if label == 'Soal/referensi'][:info['question_kept']]
            if len(prompt_positions) != info['question_kept'] or not prompt_positions or not positions:
                raise ValueError('Token soal dan jawaban harus tersedia untuk attention.')
            offset = width - len(segments) if padding_side == 'left' else 0
            question.append([i - offset in prompt_positions for i in range(width)])
            answer.append([i - offset in positions for i in range(width)])
    return (torch.tensor(question, dtype=torch.bool, device=device),
            torch.tensor(answer, dtype=torch.bool, device=device))


def correlated_trait_pairs(labels, threshold):
    # Correlation uses only the training partition of the current fit.
    centered = labels.astype(np.float64) - labels.mean(axis=0)
    norm = np.linalg.norm(centered, axis=0)
    denominators = np.outer(norm, norm)
    corr = np.divide(centered.T @ centered, denominators,
                     out=np.zeros_like(denominators), where=denominators > 0)
    return np.nonzero(np.triu(corr >= threshold, k=1))


def relation_error(prediction, targets, pairs):
    left, right = pairs
    if left.numel() == 0:
        return prediction.new_zeros(())
    error = prediction - targets
    return (error[:, left] - error[:, right]).square().mean()


class MultiTraitModel(nn.Module):
    """E2 reset: last-layer CLS, BGE projection, shared four-output MLP."""

    def __init__(self, scenario, roberta, s3_mlp_dropout=0.2, scalar_mix_layers=4, gamma_initial=1.0):
        super().__init__()
        self.scenario, self.roberta = scenario, roberta
        self.roberta.requires_grad_(True)
        self.roberta_norm = nn.LayerNorm(768)
        self.projection = nn.Sequential(nn.Linear(1024, 256), nn.LayerNorm(256))
        self.head = nn.Sequential(
            nn.Linear(1024, 256),
            nn.GELU(),
            nn.Dropout(s3_mlp_dropout),
            nn.Linear(256, 4),
        )

        # Preserve the shared E1 initialization and post-initialization RNG state.
        # Replacing self.head unregisters the temporary shared E1 scoring layers.
        with torch.random.fork_rng(devices=[]):
            self.head = PromptAwareHeads(s3_mlp_dropout)
        self.scalar_mix = nn.ModuleDict({
            name: ScalarMix(scalar_mix_layers, gamma_initial) for name, *_ in TRAITS
        })


    def forward(self, windows, mapping, n_essays, bge, question_mask, answer_mask, return_features=False):
        encoded = self.roberta(**windows, output_hidden_states=True, return_dict=True)
        projected = self.projection(bge)
        states, features = [], {}
        for index, (name, *_) in enumerate(TRAITS):
            hidden = self.scalar_mix[name](encoded.hidden_states, windows['attention_mask'])
            cls = mean_by_essay(hidden[:, 0, :], mapping, n_essays)
            fusion = torch.cat([self.roberta_norm(cls), projected], dim=1)
            states.append(self.head.trait_state(fusion, hidden, question_mask, answer_mask, mapping, index))
            if return_features:
                features[f'roberta_{name}'] = cls
                features[f'concat_{name}'] = fusion
        prediction = self.head.finish(states)
        if return_features:
            features.update(bge=bge, projection=projected)
            return prediction, features
        return prediction


### `Engine`

Siapkan input batch dan kembalikan prediksi dari skala normalisasi ke poin tiap trait.

In [ ]:
class Engine:
    """Jalankan tiga teks melalui model fusion S3 yang sama."""
    def __init__(self, model, texts, bge, cfg, device, roberta_tokenizer=None, contexts=None):
        if model.scenario != 'S3':
            raise ValueError('Engine E43 hanya menjalankan S3.')
        self.model, self.texts, self.bge, self.cfg, self.device = model, texts, bge, cfg, device
        self.contexts = contexts
        if contexts is None or len(contexts) != len(texts):
            raise ValueError('Setiap jawaban memerlukan satu pasangan soal dan referensi.')
        self.tokenizer = roberta_tokenizer or tokenizer(cfg, 'roberta')

    def forward(self, indices, return_features=False):
        ids = [int(i) for i in indices]
        features = self.bge[ids].to(self.device)
        windows, mapping, details = pair_batch_windows(
            [self.texts[i] for i in ids], [self.contexts[i] for i in ids],
            self.tokenizer, self.device, self.cfg['roberta_max_length'],
            self.cfg['roberta_overlap'], self.cfg)
        question_mask, answer_mask = prompt_answer_masks(
            details, windows['input_ids'].shape[1], self.tokenizer.padding_side, self.device)
        return self.model(question_mask=question_mask, answer_mask=answer_mask, windows=windows, mapping=mapping, n_essays=len(ids),
                          bge=features, return_features=return_features)

    def predict(self, indices):
        # Mode prediksi: nonaktifkan dropout. Pembacaan di bawah memakai no_grad.
        self.model.eval()
        values = []
        # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
        with torch.no_grad():
            for start in range(0, len(indices), self.cfg['micro_batch']):
                ids = indices[start:start+self.cfg['micro_batch']]
                values.append(self.forward(ids).float().cpu().numpy())
        scaled = np.concatenate(values)
        # Kembalikan setiap keluaran ke satuan poin trait.
        return scaled * SCORE_SPAN + SCORE_MIN


## 13 · Inisialisasi model dan spesifikasi

### `seed_everything`

Atur pengacakan training pada awal setiap fit mengikuti seed baseline sederhana.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


### `make_model`

Buat model baru untuk setiap fold dan refit.

In [ ]:
def make_model(scenario, cfg):
    return MultiTraitModel(scenario, encoder(cfg,'roberta'),
                           s3_mlp_dropout=cfg['s3_mlp_dropout'],
                           scalar_mix_layers=cfg['scalar_mix_layers'],
                           gamma_initial=cfg['scalar_mix_gamma_initial'])


### architecture

Cell ini menjalankan bagian architecture.

In [ ]:
display(pd.DataFrame([
    ['ScalarMix RoBERTa', '4 layer per trait · softmax awal 0,25; gamma awal 1'],
    ['Attention soal–jawaban', 'Isi, kebahasaan, mekanik seperti E25'],
    ['Attention antartrait', 'E25'],
    ['BGE frozen → proyeksi', '1024 → 256'],
    ['Concat → empat head', '1024 → 64 → 1 per trait'],
    ['Tambahan parameter', '20 skalar terlatih; +15 dibanding E28'],
], columns=['Bagian', 'Dimensi / setelan']))


## 14 · Metrik evaluasi

RMSE dan Pearson memakai prediksi mentah. QWK dan Gwet AC2 memakai kategori terdekat.

In [ ]:
import warnings
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score

def ordinal_prediction(values, trait):
    cats=np.asarray(trait['categories'],dtype=float)
    raw=np.asarray(values,dtype=float)
    if not np.isfinite(raw).all():raise ValueError('Prediksi nonfinite')
    a=np.clip(raw,trait['min'],trait['max'])
    # argmin memilih kategori pertama (lebih rendah) ketika jarak sama.
    return cats[np.abs(a[...,None]-cats).argmin(-1)].astype(int)

def gwet_ac2(teacher, predicted, categories):
    a=np.asarray(teacher);b=np.asarray(predicted);c=np.asarray(categories,dtype=float)
    if len(a)!=len(b) or len(a)<2: raise ValueError('AC2 perlu minimal dua pasangan')
    if not np.isin(a,c).all() or not np.isin(b,c).all(): raise ValueError('Kategori AC2 tidak sah')
    q=len(c); weights=1-((c[:,None]-c[None,:])/(c.max()-c.min()))**2
    ai=np.searchsorted(c,a); bi=np.searchsorted(c,b)
    pa=weights[ai,bi].mean()
    pi=(np.bincount(ai,minlength=q)+np.bincount(bi,minlength=q))/(2*len(a))
    pe=weights.sum()*np.sum(pi*(1-pi))/(q*(q-1))
    return float((pa-pe)/(1-pe))

def evaluate(teacher, predicted, rubric):
    y=np.asarray(teacher,dtype=float);p=np.asarray(predicted,dtype=float)
    if y.shape!=p.shape or y.ndim!=2 or y.shape[1]!=4: raise ValueError('Metrik memerlukan [N,4] yang sama')
    if not np.isfinite(y).all() or not np.isfinite(p).all(): raise ValueError('Label/prediksi nonfinite')
    result={}
    for j,t in enumerate(rubric['traits']):
        a,b=y[:,j],p[:,j]; cats=t['categories']
        if not np.isin(a,cats).all(): raise ValueError('Label harus kategori guru yang sah')
        o=ordinal_prediction(b,t); reasons={}
        rmse=float(np.sqrt(np.mean((a-b)**2))) if len(a) else None
        pearson=None
        if len(a)>=2 and np.ptp(a)>0 and np.ptp(b)>0: pearson=float(pearsonr(a,b).statistic)
        else: reasons['Pearson']='Minimal dua pasangan dan kedua input tidak konstan.'
        qwk=None;ac2=None
        if len(a)>=2:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore'); v=cohen_kappa_score(a,o,weights='quadratic',labels=cats)
            if np.isfinite(v): qwk=float(v)
            else: reasons['QWK']='Expected disagreement nol; koefisien undefined.'
            ac2=gwet_ac2(a,o,cats)
        else: reasons.update({'QWK':'Minimal dua pasangan.','AC2':'Minimal dua pasangan.'})
        result[t['name']]={'RMSE':rmse,'Pearson':pearson,'QWK':qwk,'AC2':ac2,
             'out_of_range_fraction':float(((b<t['min']) | (b>t['max'])).mean()) if len(b) else None,
             'undefined_reasons':reasons}
    return result


### `metrics` dan `predictions_array`

Periksa bentuk empat skor, lalu hitung metrik per trait. Pemilihan epoch memakai RMSE dari seluruh skor validasi pada skala asli.

In [ ]:
def metrics(teacher, prediction):
    """RMSE validasi pada seluruh pasangan skor asli, serta metrik per trait."""
    per_trait = evaluate(teacher, prediction, RUBRIC)
    residual = np.asarray(teacher, dtype=float) - np.asarray(prediction, dtype=float)
    return {'RMSE': float(np.sqrt(np.mean(residual ** 2))), 'traits': per_trait}


def predictions_array(values, indices):
    array = np.asarray(values, dtype=float)
    if array.shape != (len(indices), len(TRAITS)) or not np.isfinite(array).all():
        raise ValueError('Prediksi harus finite dan berukuran [jumlah jawaban, 4].')
    return array


## 15 · Learnable Code — optimizer, loss, dan satu fit

### `optimizer_groups`

Kelompokkan parameter dengan LR RoBERTa 5e-6 dan LR komponen lain 1e-4.

In [ ]:
def optimizer_groups(model, cfg):
    """Lima kelompok optimizer S3; semua parameter trainable tercakup tepat sekali."""
    buckets = {'roberta': [], 'projection': [], 'head': [], 'pooling_norm': []}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith('roberta.'):
            group = 'roberta'
        elif name.startswith('projection.'):
            group = 'projection'
        elif name.startswith('head.'):
            group = 'head'
        else:
            group = 'pooling_norm'
        buckets[group].append(parameter)
    rates = {'roberta': cfg['s3_encoder_lr'],
             'projection': cfg['projection_lr'],
             'head': cfg['s3_mlp_lr'],
             'pooling_norm': cfg['s3_layers_lr']}
    result = [{'name': name, 'params': params, 'lr': rates[name]} for name, params in buckets.items() if params]
    ids = [id(p) for group in result for p in group['params']]
    expected = {id(p) for p in model.parameters() if p.requires_grad}
    if len(ids) != len(set(ids)) or set(ids) != expected:
        raise ValueError('Optimizer harus mencakup setiap parameter trainable tepat sekali.')
    return result


### `fit`

Latih satu model dengan AdamW, MSE target skala 0–1, serta pencatatan loss relasi antartrait. Saat CV, pilih epoch dengan RMSE validasi terkecil.

In [ ]:
def choose_checkpoints(history, predictions_by_epoch, margin):
    # Early stopping dan checkpoint A memakai RMSE, seperti pada E19.
    evaluated = [row for row in history if 'validation' in row]
    best_rmse = min(evaluated, key=lambda row: (row['validation']['RMSE'], row['epoch']))
    rmse_limit = best_rmse['validation']['RMSE'] + margin
    candidates = []
    for row in evaluated:
        validation = row['validation']
        qwk = [validation['traits'][name]['QWK'] for name, *_ in TRAITS]
        if validation['RMSE'] <= rmse_limit + 1e-12 and all(value is not None for value in qwk):
            candidates.append((row, float(np.mean(qwk))))
    # Jika QWK suatu trait tidak terdefinisi untuk semua kandidat, pakai A.
    best_qwk = best_rmse
    if candidates:
        best_qwk, _ = min(
            candidates, key=lambda item: (-item[1], item[0]['validation']['RMSE'], item[0]['epoch']),
        )

    return {
        choice: {'epoch': row['epoch'], 'prediction': predictions_by_epoch[row['epoch']].tolist()}
        for choice, row in [('A', best_rmse), ('B', best_qwk)]
    }, len(candidates)


def organization_loss_weights(labels, ratio):
    # Bobot ditentukan hanya dari label organisasi di bagian training fold.
    scores = np.asarray(labels)
    if scores.ndim != 1 or not len(scores) or not np.isin(scores, (1, 2)).all():
        raise ValueError('Label organisasi untuk bobot loss harus berisi skor 1 atau 2.')
    if not math.isfinite(ratio) or ratio < 1:
        raise ValueError('Rasio bobot organisasi harus berhingga dan minimal 1.')

    n_low = int(np.sum(scores == 1))
    n_high = len(scores) - n_low
    average = (ratio * n_low + n_high) / len(scores)
    return [
        {'skor': 1, 'n_training': n_low, 'bobot_loss': ratio / average},
        {'skor': 2, 'n_training': n_high, 'bobot_loss': 1 / average},
    ]


def fit(scenario, training, validation, bundle, bge, cfg, epochs, seed, candidate_weights):
    pair_indices = correlated_trait_pairs(bundle.y[training], cfg['trait_corr_threshold'])
    active_pairs = [[TRAITS[i][0], TRAITS[j][0]] for i, j in zip(*pair_indices)]
    seed_everything(seed)
    device = device_name(cfg)
    pairs = tuple(torch.as_tensor(indices, device=device) for indices in pair_indices)
    print(f'Pasangan trait aktif: {active_pairs}', flush=True)
    org_column = next(i for i, (name, *_) in enumerate(TRAITS) if name == 'organisasi')
    org_weights = organization_loss_weights(bundle.y[training, org_column], cfg['organization_loss_ratio'])
    # Untuk rubrik organisasi 1–2, target ternormalisasi 0 berarti skor asli 1.
    weight_low, weight_high = (row['bobot_loss'] for row in org_weights)
    print(f'Bobot organisasi per skor: {org_weights}', flush=True)
    isi_column = next(i for i, (name, *_) in enumerate(TRAITS) if name == 'isi')
    isi_labels = bundle.y[training, isi_column]
    n_low = int(np.sum(isi_labels == 1))
    ratio = cfg['isi_loss_ratio']
    mean_weight = 1.0 + (ratio - 1.0) * n_low / len(training)
    isi_low, isi_other = ratio / mean_weight, 1.0 / mean_weight
    isi_weights = [
        {'skor': '1', 'n_training': n_low, 'bobot_loss': isi_low},
        {'skor': '2–4', 'n_training': len(training) - n_low, 'bobot_loss': isi_other},
    ]
    print(f'Bobot loss isi dari training fold: {isi_weights}', flush=True)
    # Buat model baru untuk fold ini, lalu tempatkan di GPU atau CPU yang tersedia.
    model = make_model(scenario, cfg).to(device)
    engine = Engine(model, bundle.texts, bge, cfg, device, contexts=contexts_from_bundle(bundle))
    groups = optimizer_groups(model, cfg)
    learning_rates = {group['name']: group['lr'] for group in groups}
    # AdamW memperbarui kelompok parameter sesuai learning rate masing-masing.
    optimizer = torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader([int(i) for i in training], batch_size=cfg['effective_batch'],
                        shuffle=True, generator=generator, num_workers=0)
    total_steps = len(loader)*epochs
    warmup = min(int(cfg['warmup_ratio']*total_steps), max(0, total_steps-1))
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total_steps)
    use_cuda = device == 'cuda'
    dtype = torch.bfloat16 if use_cuda and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda and dtype == torch.float16)
    selected_parameters = {}
    for name, parameter in model.named_parameters():
        group = name.split('.')[0]
        if parameter.requires_grad and group not in selected_parameters:
            selected_parameters[group] = (name, parameter.detach().reshape(-1)[:32].cpu().clone())
    best_state, best_epoch, best_rmse = None, 0, float('inf')
    patience = cfg['early_stopping_patience']
    without_improvement, early_stopped = 0, False
    predictions_by_epoch = {}
    history, active_steps, optimizer_steps, skipped_steps = [], 0, 0, 0
    for epoch in range(1, epochs+1):
        model.train()
        loss_total = mse_total = weighted_total = relation_total = 0.
        for effective in loader:
            batch = effective.tolist()
            optimizer.zero_grad(set_to_none=True)
            for start in range(0, len(batch), cfg['micro_batch']):
                ids = batch[start:start+cfg['micro_batch']]
                # Normalisasi per trait memberi bobot sama pada empat dimensi loss.
                targets = torch.as_tensor((bundle.y[ids] - SCORE_MIN) / SCORE_SPAN,
                                          dtype=torch.float32, device=device)
                with torch.autocast(device_type=device, dtype=dtype, enabled=use_cuda):
                    prediction = engine.forward(ids).float()
                    if prediction.shape != targets.shape:
                        raise ValueError('Shape prediksi/target harus [B,4], tanpa broadcasting.')
                    # MSE mengukur selisih kuadrat antara prediksi dan target; optimizer berusaha memperkecilnya.
                    squared_error = (prediction - targets).square()
                    mse = squared_error.mean()
                    weights = torch.ones_like(squared_error)
                    weights[:, org_column] = torch.where(
                        targets[:, org_column] == 0, weight_low, weight_high
                    )
                    weights[:, isi_column] = torch.where(
                        targets[:, isi_column] == 0, isi_low, isi_other
                    )
                    weighted_mse = (squared_error * weights).mean()
                    relation = relation_error(prediction, targets, pairs)
                    loss = weighted_mse + cfg['trait_relation_weight'] * relation
                if not torch.isfinite(loss):
                    raise RuntimeError('Loss nonfinite; run dihentikan, tidak diklaim berhasil.')
                # Hitung gradien dan bobot microbatch sesuai ukuran batch efektif aktual.
                scaler.scale(loss*(len(ids)/len(batch))).backward()
                loss_total += loss.item()*len(ids)
                mse_total += mse.item()*len(ids)
                weighted_total += weighted_mse.item()*len(ids)
                relation_total += relation.item()*len(ids)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['gradient_clip'])
            lr_active = any(g['lr'] > 0 for g in optimizer.param_groups)
            previous_scale = scaler.get_scale()
            # Perbarui bobot; GradScaler bisa melewati langkah yang overflow pada AMP.
            scaler.step(optimizer)
            scaler.update()
            skipped = scaler.is_enabled() and scaler.get_scale() < previous_scale
            if skipped:
                skipped_steps += 1
            else:
                active_steps += int(lr_active)
                scheduler.step()
                optimizer_steps += 1
        row = {'epoch': epoch, 'train_mse_scaled': mse_total/len(training),
               'train_weighted_mse_scaled': weighted_total/len(training),
               'train_relation_loss': relation_total/len(training),
               'train_total_loss': loss_total/len(training), 'active_trait_pairs': active_pairs,
               'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
               'effective_batch_target': cfg['effective_batch'], 'micro_batch': cfg['micro_batch']}
        if validation is not None:
            predicted = engine.predict(validation)
            predictions_by_epoch[epoch] = predicted.copy()
            row['validation'] = metrics(bundle.y[validation], predicted)
            # Simpan epoch dengan RMSE validasi terkecil; data uji akhir tidak ikut memilih epoch.
            if row['validation']['RMSE'] < best_rmse:
                best_rmse, best_epoch = row['validation']['RMSE'], epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                without_improvement = 0
            else:
                without_improvement += 1
            candidate_weights.observe(model, row, best_rmse)
            row.update(best_epoch_so_far=best_epoch, epochs_without_improvement=without_improvement)
            message = (f" | RMSE validasi: {row['validation']['RMSE']:.4f} poin"
                       f" | terbaik: epoch {best_epoch}, {best_rmse:.4f} poin"
                       f" | tanpa perbaikan: {without_improvement}/{patience}")
        else:
            best_epoch = epoch
            message = ' | pelatihan final; data uji belum dibuka'
        history.append(row)
        print(f'{scenario} · epoch {epoch}/{epochs} · total loss latih: {row["train_total_loss"]:.6f}'+message, flush=True)
        # Hanya fold validasi yang boleh menghentikan latihan; data uji tidak dibaca.
        if validation is not None and without_improvement >= patience and epoch < epochs:
            early_stopped = True
        if early_stopped:
            print(f'{scenario}: early stopping setelah {epoch} epoch; RMSE validasi tidak membaik selama {patience} epoch. '
                  f'Bobot epoch {best_epoch} dipulihkan.', flush=True)
            break
    if active_steps == 0:
        raise RuntimeError('Tidak ada pembaruan dengan LR aktif.')
    named = dict(model.named_parameters())
    changed = {group: not torch.equal(before, named[name].detach().reshape(-1)[:32].cpu())
               for group, (name, before) in selected_parameters.items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    info = {'scenario': scenario, 'best_epoch': best_epoch, 'epochs_completed': len(history),
            'max_epochs': epochs, 'early_stopping_patience': patience if validation is not None else None,
            'early_stopped': early_stopped, 'optimizer_learning_rates': learning_rates,
            'best_validation_rmse': best_rmse if validation is not None else None,
            'stop_reason': ('RMSE validasi tidak membaik selama batas kesabaran' if early_stopped else
                            'Batas maksimum epoch tercapai' if validation is not None else
                            'Jumlah epoch final yang dipilih dari CV selesai'),
            'history': history, 'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
            'parameter_samples_changed': changed, 'BGE_encoder_trained': False,
            'organization_loss_weights': org_weights, 'isi_loss_weights': isi_weights,
            'selection': 'minimum validation RMSE in original score units' if validation is not None else 'fixed refit epochs selected from CV'}
    # Bobot diambil SESUDAH best_state dipulihkan; sesuai model yang dievaluasi.
    choices, eligible = choose_checkpoints(history, predictions_by_epoch, cfg['checkpoint_rmse_margin'])
    if choices['A']['epoch'] != best_epoch:
        raise ValueError('Epoch A berbeda dari checkpoint RMSE yang dipulihkan.')
    info['eligible_epochs_for_B'] = eligible
    return model, engine, info, choices


## 16 · Mulai validasi dan siapkan penyimpanan

Definisi di bawah menyiapkan tabel dan ZIP. Lima cell sesudahnya menjalankan
training satu fold per cell, menampilkan metrik, lalu menyimpan hasilnya.
Jalankan kembali dari atas untuk memulihkan progres dari Drive. Fold yang sempat
putus di tengah dilatih lagi dari awal.

In [ ]:
def excel_value(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False, default=str)
    if isinstance(value, float) and not math.isfinite(value):
        return None  # Tidak tercakup/undefined: tetap kosong, bukan nol.
    if value is pd.NA:
        return None
    return value


In [ ]:
def write_excel(path, sheets, metadata):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    book = Workbook(write_only=True)
    used = set()
    counts = {}
    all_sheets = {'Info': pd.DataFrame([{'field': k, 'value': v} for k, v in metadata.items()]), **sheets}
    for name, frame in all_sheets.items():
        frame = frame if isinstance(frame, pd.DataFrame) else pd.DataFrame(frame)
        if len(frame.columns) > 16384:
            raise ValueError('Terlalu banyak kolom untuk Excel; tidak dipotong diam-diam.')
        headers = [str(x) for x in frame.columns]
        chunks = max(1, math.ceil(len(frame)/1048575))
        for part in range(chunks):
            base = re.sub(r'[\\/*?:\[\]]', '_', name)
            suffix = f'_{part+1}' if chunks > 1 else ''
            title = base[:31-len(suffix)]+suffix
            if title in used:
                raise ValueError('Nama sheet bertabrakan setelah dipendekkan: '+title)
            used.add(title)
            sheet = book.create_sheet(title)
            sheet.freeze_panes = 'B2'
            heading = []
            for text in headers:
                cell = WriteOnlyCell(sheet, value=text)
                cell.font = Font(bold=True, color='FFFFFF')
                cell.fill = PatternFill('solid', fgColor='172035')
                cell.data_type = 's'
                heading.append(cell)
            sheet.append(heading)
            subset = frame.iloc[part*1048575:(part+1)*1048575]
            for row in subset.itertuples(index=False, name=None):
                values = []
                for value in row:
                    value = excel_value(value)
                    if isinstance(value, str):
                        if len(value) > 32767:
                            raise ValueError('Teks melebihi batas satu sel Excel; ekspor dihentikan agar tidak terpotong.')
                        cell = WriteOnlyCell(sheet, value=value)
                        cell.data_type = 's'  # Teks siswa '=...' tidak dijalankan sebagai formula.
                        values.append(cell)
                    else:
                        values.append(value)
                sheet.append(values)
            counts[title] = {'rows': len(subset), 'columns': len(headers)}
    temporary = path.with_name(path.stem+'.pending.xlsx')
    book.save(temporary)
    temporary.replace(path)
    return counts


def export_excel_zip(path, groups, metadata, max_bytes=30_000_000):
    """Simpan workbook utuh per bagian. Pecah lagi jika ukuran aktual melewati batas."""
    from tempfile import TemporaryDirectory

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    inventory = []
    with TemporaryDirectory(prefix='xlsx_', dir=path.parent) as temporary:
        staging = Path(temporary)

        def write_part(stem, tables, relative_part='', row_start=0):
            candidate = staging / f'{stem}{relative_part}.xlsx'
            counts = write_excel(candidate, tables, metadata)
            size = candidate.stat().st_size
            if size < max_bytes:
                inventory.append({
                    'file': candidate.name, 'bytes': size, 'sha256': digest(candidate),
                    'sheet_sizes': counts,
                    'tables': [
                        {'name': name, 'row_start': row_start, 'rows': len(frame)}
                        for name, frame in tables.items()
                    ],
                })
                return

            candidate.unlink()
            if len(tables) > 1:
                entries = list(tables.items())
                middle = len(entries) // 2
                write_part(stem, dict(entries[:middle]), relative_part + '_1', row_start)
                write_part(stem, dict(entries[middle:]), relative_part + '_2', row_start)
                return

            name, frame = next(iter(tables.items()))
            if len(frame) <= 1:
                raise ValueError(f'Sheet {name} tidak dapat dipisah lagi agar di bawah {max_bytes:,} byte.')
            middle = len(frame) // 2
            write_part(stem, {name: frame.iloc[:middle]}, relative_part + '_1', row_start)
            write_part(stem, {name: frame.iloc[middle:]}, relative_part + '_2', row_start + middle)

        for stem, tables in groups.items():
            if tables:
                write_part(stem, tables)

        # XLSX sudah dikompresi. Simpan langsung tanpa mengulang kompresi ZIP.
        pending = staging / 'hasil.zip'
        folder = path.stem
        manifest_name = f'{folder}/manifest.json'
        with zipfile.ZipFile(pending, 'w', compression=zipfile.ZIP_STORED) as archive:
            for entry in inventory:
                archive.write(staging / entry['file'], f"{folder}/{entry['file']}")
            archive.writestr(manifest_name, json.dumps({
                'xlsx_size_limit_bytes_exclusive': max_bytes,
                'table_row_start_is_zero_based': True,
                'workbooks': inventory,
            }, ensure_ascii=False, indent=2))

        with zipfile.ZipFile(pending) as archive:
            sizes = {item.filename: item.file_size for item in archive.infolist()}
            expected = {f"{folder}/{entry['file']}": entry['bytes'] for entry in inventory}
            if (set(sizes) != {*expected, manifest_name}
                or any(sizes[name] != size or size >= max_bytes for name, size in expected.items())
                or archive.testzip() is not None):
                raise IOError('Isi ZIP gagal diverifikasi.')
        pending.replace(path)
    return inventory


In [ ]:
def lowest_score_rows(bundle, indices, prediction, stage):
    scores = predictions_array(prediction, indices)
    rows = []
    for column, trait in enumerate(RUBRIC['traits']):
        predicted = ordinal_prediction(scores[:, column], trait)
        lowest = bundle.y[indices, column] == trait['min']
        total = int(lowest.sum())
        correct = int(np.sum(lowest & (predicted == trait['min'])))
        rows.append({
            'bagian': stage, 'trait': trait['name'], 'skor_terendah': trait['min'],
            'n_label_terendah': total, 'n_terdeteksi': correct,
            'proporsi_terdeteksi': correct / total if total else None,
        })
    return rows


def trait_rows(values, **keys):
    return [{**keys, 'trait': name,
             'RMSE': stats['RMSE'], 'Pearson': stats['Pearson'],
             'QWK': stats['QWK'], 'Gwet AC2': stats['AC2'],
             'proporsi_di_luar_skala': stats['out_of_range_fraction'],
             'alasan_metrik_kosong': stats['undefined_reasons']}
            for name, stats in values['traits'].items()]


def prediction_rows(bundle, indices, prediction, **keys):
    scores = predictions_array(prediction, indices)
    categories = np.column_stack([
        ordinal_prediction(scores[:, i], trait) for i, trait in enumerate(RUBRIC['traits'])
    ])
    rows = []
    for j, i in enumerate(indices):
        source = bundle.rows[i]
        row = {**keys, 'id_jawaban': source['row_id'], 'baris_excel': source['source_row'],
               'no_soal': source['response_number'], 'kelompok': int(bundle.groups[i])}
        for col, (name, _, _, _) in enumerate(TRAITS):
            actual, raw = bundle.y[i, col], scores[j, col]
            row.update({f'{name}_label': int(actual), f'{name}_prediksi_mentah': float(raw),
                        f'{name}_prediksi_kategori': int(categories[j, col]),
                        f'{name}_error_abs': float(abs(raw - actual))})
        rows.append(row)
    return rows


def history_rows(info, **keys):
    return [{**keys, **{k: v for k, v in row.items() if k != 'validation'},
             'validation_RMSE': row.get('validation', {}).get('RMSE')}
            for row in info['history']]

In [ ]:
def checkpoint_tables():
    per_fold, per_trait, predictions, low_scores, epoch_metrics = [], [], [], [], []
    for number, result in sorted(fold_results.items()):
        valid = result['identity']['validation']
        for choice, checkpoint in result['checkpoints'].items():
            prediction = checkpoint['prediction']
            evaluation = metrics(bundle.y[valid], prediction)
            per_fold.append({
                'fold': number, 'pilihan': choice, 'epoch': checkpoint['epoch'],
                'RMSE': evaluation['RMSE'],
                'macro_QWK': mean_defined_qwk(evaluation),
            })
            tags = {'pilihan': choice, 'fold': number}
            per_trait.extend(trait_rows(evaluation, **tags, n=len(valid)))
            predictions.extend(prediction_rows(bundle, valid, prediction, **tags))
            low_scores.extend(checkpoint_score_errors(valid, prediction, **tags))
        for row in result['training']['history']:
            epoch_metrics.extend(trait_rows(row['validation'], fold=number, epoch=row['epoch'], n=len(valid)))

    return {
        'Pilihan Checkpoint': pd.DataFrame(per_fold),
        'A B per Fold dan Trait': pd.DataFrame(per_trait),
        'Prediksi OOF A B': pd.DataFrame(predictions),
        'Kesalahan Skor Rendah': pd.DataFrame(low_scores),
    }, pd.DataFrame(epoch_metrics)


def mean_defined_qwk(evaluation):
    values = [stats['QWK'] for stats in evaluation['traits'].values()]
    return float(np.mean(values)) if all(x is not None for x in values) else None


def checkpoint_score_errors(indices, predicted, **keys):
    errors = []
    for column, trait in enumerate(RUBRIC['traits']):
        labels = bundle.y[indices, column]
        category = ordinal_prediction(np.asarray(predicted)[:, column], trait)
        errors.append({
            **keys, 'trait': trait['name'], 'jumlah label 1': int(np.sum(labels == 1)),
            '1 benar': int(np.sum((labels == 1) & (category == 1))),
            '1 salah jadi 2': int(np.sum((labels == 1) & (category == 2))),
            '2 salah jadi 1': int(np.sum((labels == 2) & (category == 1))),
        })
    return errors



def isi_counts(indices, prediction):
    column = next(i for i, (name, *_) in enumerate(TRAITS) if name == 'isi')
    labels = bundle.y[indices, column]
    classes = ordinal_prediction(np.asarray(prediction)[:, column], RUBRIC['traits'][column])
    low = labels == 1
    return {
        'jumlah jawaban': len(indices),
        'label 1': int(low.sum()),
        'label 1 terdeteksi': int((low & (classes == 1)).sum()),
        'label 2–4 salah jadi 1': int((~low & (classes == 1)).sum()),
    }



def isi_audit_tables():
    comparisons, weights = [], []
    for number, result in sorted(fold_results.items()):
        valid = result['identity']['validation']
        info = result['training']
        prediction = result['checkpoints']['B']['prediction']
        comparisons.append({'fold': number, 'bagian': 'latih', **info['isi_latih_B']})
        comparisons.append({'fold': number, 'bagian': 'validasi', **isi_counts(valid, prediction)})
        weights.extend({'fold': number, **row} for row in info['isi_loss_weights'])
    return pd.DataFrame(comparisons), pd.DataFrame(weights)




def compare_reference(worksheets, reference):
    current = worksheets['OOF A B per Trait'].query("pilihan == 'B'").set_index('trait')
    rows = []
    for trait_row in reference['metrics']:
        trait = trait_row['trait']
        for metric in ('RMSE', 'Pearson', 'QWK', 'Gwet AC2'):
            value = current.loc[trait, metric]
            baseline = trait_row[metric]
            delta = None if pd.isna(value) or pd.isna(baseline) else value - baseline
            rows.append({'trait': trait, 'metrik': metric, 'E29 (B)': value,
                         'Pembanding (B)': baseline, 'delta E29 minus pembanding': delta})
    return pd.DataFrame(rows)


def export_validation(path, final=False):
    summaries, per_trait, predictions, history, low_scores = [], [], [], [], []
    indices, scores = [], []
    for number, result in sorted(fold_results.items()):
        valid, predicted = result['identity']['validation'], result['prediction']
        info = result['training']
        evaluation = metrics(bundle.y[valid], predicted)
        summaries.append({
            'fold': number, 'RMSE': evaluation['RMSE'],
            'epoch_terpilih': info['best_epoch'], 'epoch_dijalankan': info['epochs_completed'],
            'menit_training': result['seconds'] / 60,
        })
        per_trait.extend(trait_rows(evaluation, fold=number, epoch_terpilih=info['best_epoch'], n=len(valid)))
        predictions.extend(prediction_rows(bundle, valid, predicted, fold=number))
        history.extend(history_rows(info, fold=number))
        low_scores.extend({'fold': number, **row} for row in
                          lowest_score_rows(bundle, valid, predicted, 'validasi'))
        indices.extend(valid)
        scores.extend(predicted)

    completed = len(fold_results)
    if final and sorted(indices) != sorted(manifest['development']):
        raise ValueError('ZIP final membutuhkan tepat satu prediksi untuk setiap jawaban development.')
    summary = {
        'eksperimen': cfg['experiment'], 'fold_selesai': completed,
        'jumlah_fold': len(manifest['folds']), 'jawaban_OOF': len(indices),
        'mean_cv_RMSE': float(np.mean([row['RMSE'] for row in summaries])),
        'status': 'lengkap' if final else 'progres',
    }
    worksheets = {
        'Ringkasan CV': pd.DataFrame([summary]),
        'Progres Fold': pd.DataFrame(summaries),
        'CV per Fold dan Trait': pd.DataFrame(per_trait),
        'Prediksi OOF': pd.DataFrame(predictions),
        'Skor Terendah': pd.DataFrame(low_scores),
    }
    if final:
        evaluation = metrics(bundle.y[indices], scores)
        worksheets['CV OOF per Trait'] = pd.DataFrame(trait_rows(evaluation, n=len(indices)))
        worksheets['Skor Terendah'] = pd.concat([
            worksheets['Skor Terendah'],
            pd.DataFrame(lowest_score_rows(bundle, indices, scores, 'OOF')),
        ], ignore_index=True)
    paired, epoch_metrics = checkpoint_tables()
    worksheets.update(paired)
    if final:
        pooled_rows, pooled_errors = [], []
        ab_predictions = paired['Prediksi OOF A B']
        for choice, frame in ab_predictions.groupby('pilihan', sort=True):
            label_columns = [f'{name}_label' for name, *_ in TRAITS]
            pred_columns = [f'{name}_prediksi_mentah' for name, *_ in TRAITS]
            if frame['id_jawaban'].duplicated().any() or len(frame) != len(indices):
                raise ValueError(f'Prediksi OOF {choice} harus lengkap dan unik.')
            evaluation = metrics(frame[label_columns].to_numpy(), frame[pred_columns].to_numpy())
            pooled_rows.extend(trait_rows(evaluation, pilihan=choice, n=len(indices)))
            pooled_errors.extend(checkpoint_score_errors(
                indices, frame[pred_columns].to_numpy(), pilihan=choice,
            ))
        worksheets['OOF A B per Trait'] = pd.DataFrame(pooled_rows)
        worksheets['Kesalahan Skor Rendah OOF'] = pd.DataFrame(pooled_errors)

    isi_comparisons, isi_weights = isi_audit_tables()
    worksheets['Isi 1 Latih dan Validasi B'] = isi_comparisons
    if final:
        worksheets['E29 vs E28 per Trait'] = compare_reference(worksheets, E28_REFERENCE)
        worksheets['E29 vs E25 per Trait'] = compare_reference(worksheets, E25_REFERENCE)
    export_excel_zip(path, {
        '04_Prediksi_dan_Evaluasi': worksheets,
        '05_Riwayat_Pelatihan': {
            'Riwayat Training': pd.DataFrame(history),
            'Metrik Validasi per Epoch': epoch_metrics,
            'Bobot Loss Organisasi': pd.DataFrame([
                {'fold': number, **row}
                for number, result in sorted(fold_results.items())
                for row in result['training']['organization_loss_weights']
            ]),
            'Bobot Loss Isi': isi_weights,
            'ScalarMix Checkpoint B': pd.DataFrame([{'fold': number, **row}
                for number, result in sorted(fold_results.items())
                for row in result['training']['scalar_mix_B']]),
            'Konfigurasi': pd.DataFrame(cfg.items(), columns=['parameter', 'nilai']),
        },
    }, {
        **bundle.summary, 'experiment': cfg['experiment'],
        'organization_loss_ratio': cfg['organization_loss_ratio'], 'training_seed': cfg['seed'],
        'development_rows': len(manifest['development']),
        'split_sha256': manifest['split_sha256'], 'notebook_code_sha256': NOTEBOOK_CODE_SHA256,
        'final_config': cfg, 'test_dipakai_untuk_seleksi_epoch': False,
        'test_dievaluasi': False, 'e25_seed44_source_sha256': E25_REFERENCE['source_sha256'],
        'e28_source_sha256': E28_REFERENCE['source_sha256'], 'fold_selesai': completed,
    })
    return worksheets


In [ ]:
from time import perf_counter
from tempfile import TemporaryDirectory


class CandidateWeights:
    """Keep validation candidates that could still win B, on local disk."""

    def __init__(self, directory, margin):
        self.directory = Path(directory)
        self.margin = margin
        self.candidates = {}

    def path(self, epoch):
        return self.directory / f'epoch_{epoch}.pt'

    def observe(self, model, row, best_rmse):
        limit = best_rmse + self.margin + 1e-12
        for epoch, (rmse, _) in list(self.candidates.items()):
            if rmse > limit:
                self.path(epoch).unlink()
                del self.candidates[epoch]

        evaluation = row['validation']
        qwk = mean_defined_qwk(evaluation)
        rmse = evaluation['RMSE']
        if qwk is None or rmse > limit:
            return
        if any(old_rmse <= rmse and old_qwk >= qwk
               for old_rmse, old_qwk in self.candidates.values()):
            return
        for epoch, (old_rmse, old_qwk) in list(self.candidates.items()):
            if rmse <= old_rmse and qwk >= old_qwk:
                self.path(epoch).unlink()
                del self.candidates[epoch]
        torch.save({k: value.detach().cpu() for k, value in model.state_dict().items()},
                   self.path(row['epoch']))
        self.candidates[row['epoch']] = (rmse, qwk)


def fit_and_check_isi(training, validation, seed):
    with TemporaryDirectory(prefix='aes_e23_') as directory:
        weights = CandidateWeights(directory, cfg['checkpoint_rmse_margin'])
        model, engine, info, choices = fit(
            'S3', training, validation, bundle, bge, cfg, cfg['epochs'], seed, weights,
        )
        choice = choices['B']
        if choice['epoch'] != choices['A']['epoch']:
            model.load_state_dict(torch.load(
                weights.path(choice['epoch']), map_location='cpu', weights_only=True,
            ))
        fitted_valid = predictions_array(engine.predict(validation), validation)
        if not np.allclose(fitted_valid, choice['prediction'], rtol=0, atol=1e-6):
            raise ValueError('Bobot B tidak mereproduksi prediksi validasi.')
        info['isi_latih_B'] = isi_counts(training, engine.predict(training))
        info['scalar_mix_B'] = [
            {'trait': name, **row} for name, *_ in TRAITS
            for row in model.scalar_mix[name].report(model.roberta.config.num_hidden_layers)
        ]
        del engine, model
    release()
    return info, choices, predictions_array(choices['A']['prediction'], validation)



def run_validation_fold(number):
    fold = manifest['folds'][number - 1]
    if fold['fold'] != number:
        raise ValueError(f'Nomor fold tidak sesuai: {number}.')
    training, validation = fold['train'], fold['validation']
    seed = cfg['seed'] + number
    path = output / f'fold_{number}.json'
    identity = {
        'split': manifest['split_sha256'],
        'fold': number,
        'train': training,
        'validation': validation,
        'seed': seed,
    }
    if path.exists():
        result = json.loads(path.read_text())
        if result['identity'] != identity:
            raise ValueError(f'Cache fold {number} memakai pembagian atau seed berbeda.')
        print(f'Fold {number}/{len(manifest["folds"])} · hasil tersimpan dipakai ulang.')
    else:
        print(f'Fold {number}/{len(manifest["folds"])} · seed {seed} · '
              f'{len(training)} jawaban latih, {len(validation)} validasi', flush=True)
        started = perf_counter()
        info, checkpoints, prediction = fit_and_check_isi(training, validation, seed)
        result = {'identity': identity, 'prediction': prediction.tolist(),
                  'training': info, 'checkpoints': checkpoints, 'seconds': perf_counter() - started}
        assert_best_epoch(result)
        dump(path, result)
    assert_best_epoch(result)
    return result


def assert_best_epoch(result):
    valid = result['identity']['validation']
    prediction = predictions_array(result['prediction'], valid)
    score = metrics(bundle.y[valid], prediction)['RMSE']
    if not math.isclose(score, result['training']['best_validation_rmse'], abs_tol=1e-6):
        raise ValueError('Prediksi tersimpan tidak cocok dengan RMSE epoch terpilih.')
    history = {row['epoch']: row['validation'] for row in result['training']['history']}
    for choice, checkpoint in result['checkpoints'].items():
        selected = predictions_array(checkpoint['prediction'], valid)
        expected = history[checkpoint['epoch']]
        actual = metrics(bundle.y[valid], selected)
        if not math.isclose(actual['RMSE'], expected['RMSE'], abs_tol=1e-6):
            raise ValueError(f'Prediksi checkpoint {choice} berbeda dari riwayat epoch.')
        if mean_defined_qwk(actual) != mean_defined_qwk(expected):
            raise ValueError(f'QWK checkpoint {choice} berbeda dari riwayat epoch.')


def show_fold(number):
    if not model_stage_ready(f'Fold {number}', ('bge', 'fold_results')):
        return
    result = run_validation_fold(number)
    fold_results[number] = result
    valid, prediction = result['identity']['validation'], result['prediction']
    info = result['training']
    result_metrics = metrics(bundle.y[valid], prediction)
    print(f'Fold {number} tersimpan · terbaik epoch {info["best_epoch"]} '
          f'dari {info["epochs_completed"]} · RMSE {result_metrics["RMSE"]:.4f} · '
          f'{result["seconds"] / 60:.1f} menit saat training pertama.')
    display(pd.DataFrame(trait_rows(result_metrics, n=len(valid))).round(4))
    options = []
    for choice, checkpoint in result['checkpoints'].items():
        evaluation = metrics(bundle.y[valid], checkpoint['prediction'])
        org = next(row for row in checkpoint_score_errors(valid, checkpoint['prediction'])
                   if row['trait'] == 'organisasi')
        options.append({'pilihan': choice, 'epoch': checkpoint['epoch'],
                        'RMSE': evaluation['RMSE'], 'macro_QWK': mean_defined_qwk(evaluation),
                        'QWK organisasi': evaluation['traits']['organisasi']['QWK'],
                        'skor organisasi 1 benar': org['1 benar'],
                        'skor organisasi 2 salah jadi 1': org['2 salah jadi 1']})
    display(pd.DataFrame(options).round(4))
    display(pd.DataFrame([{'bagian': 'latih', **info['isi_latih_B']},
                          {'bagian': 'validasi', **isi_counts(valid, result['checkpoints']['B']['prediction'])}]))
    if not info['eligible_epochs_for_B']:
        print('QWK empat trait tidak lengkap pada kandidat yang memenuhi batas; B memakai A.')
    progress_path = output / 'Hasil_AES_Reset_E29_Progres_04_05.zip'
    export_validation(progress_path)
    print(f'Progres {len(fold_results)}/{len(manifest["folds"])} fold · {progress_path}')


fold_results = {}


### Fold 1 dari 5

In [ ]:
show_fold(1)

### Fold 2 dari 5

In [ ]:
show_fold(2)

### Fold 3 dari 5

In [ ]:
show_fold(3)

### Fold 4 dari 5

In [ ]:
show_fold(4)

### Fold 5 dari 5

In [ ]:
show_fold(5)

## 17 · Bandingkan checkpoint A dan B

Rekap prediksi validasi dan unduh ZIP 04–05.

In [ ]:
if model_stage_ready('Rekap validasi A dan B', ('fold_results',)):
    expected_folds = {fold['fold'] for fold in manifest['folds']}
    if set(fold_results) != expected_folds:
        print('Rekap final menunggu fold:', ', '.join(map(str, sorted(expected_folds - fold_results.keys()))))
    else:
        final_path = output / 'Hasil_AES_Reset_E29_Validasi_04_05.zip'
        sheets = export_validation(final_path, final=True)
        print('RMSE dan macro QWK · rata-rata lima fold')
        display(sheets['Pilihan Checkpoint'].groupby('pilihan')[['RMSE', 'macro_QWK']].mean().round(4))
        print('Metrik gabungan dari prediksi OOF masing-masing pilihan')
        display(sheets['OOF A B per Trait'].round(4))
        display(sheets['Kesalahan Skor Rendah OOF'])
        print('Deteksi skor isi 1 pada checkpoint B')
        display(sheets['Isi 1 Latih dan Validasi B'])
        print('Perbandingan arsitektur pada seed 44')
        print('E29 dibanding E28 · delta = E29 minus pembanding')
        display(sheets['E29 vs E28 per Trait'].round(4))
        print('E29 dibanding E25')
        display(sheets['E29 vs E25 per Trait'].round(4))
        print(f'ZIP validasi A/B · {final_path}')
        if 'google.colab' in sys.modules:
            from google.colab import files
            files.download(str(final_path))
